# Research foundations: what do my saved order books tell me?

**Question:** what visible liquidity did OKX and Bitfinex report, and how confidently
can I reconstruct it from saved inputs?

This notebook reads **OKX BTC-USDT historical L2 books** (aggregate price levels)
and **Bitfinex BTC/USD recorded live R0 books** (visible individual orders).
It uses the existing package readers. It makes **no network requests** and needs
no account or credentials. To obtain inputs first, run
[market-data-capture.ipynb](market-data-capture.ipynb), which records public feeds
and downloads the selected OKX day.

These are order books, not trades. They do not replace the
[frozen P1 trade study](../docs/research/p1-baseline.md). The sources have different
quote currencies, observation times and depth limits; their prices are not a
simultaneous arbitrage comparison. Restart the kernel and Run All in order.

Sections 9–16 add separate synthetic trade-origin, correction, eligibility, quality, manifest and snapshot and dataset command exercises
(NB 09–16). They need no Binance access and leave the saved book exploration intact.


## 1. Which implementation am I using?

The checks below require the pinned interpreter and an installed package whose
runtime files match this checkout. The table identifies this execution so that a
later result can be compared with the same code and saved inputs.


In [1]:
# standard library imports
import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import UTC, datetime, timedelta
from decimal import Decimal
from html import escape
from importlib.metadata import distributions, version
from pathlib import Path

# third party library imports
from IPython.display import HTML, display

# project library imports
import alpha_research
from alpha_research.data.live.config import CaptureConfig, CaptureRequest
from alpha_research.data.live.journal import CaptureJournal, CaptureRecord
from alpha_research.data.live.replay import CaptureReplay
from alpha_research.data.raw_objects import RawObject
from alpha_research.data.venues.bitfinex.book import BitfinexBook
from alpha_research.data.venues.okx.history_config import OKXArchiveRequest, OKXHistoryConfig
from alpha_research.data.venues.okx.history_store import OKXArchiveStore
from alpha_research.serialization import DomainCodec

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
assert (root / "configs/history/okx-btc-usdt.json").is_file(), (
    "Run from the repository root or examples/."
)
notebook_path = Path(
    os.environ.get("ALPHA_RESEARCH_NOTEBOOK_PATH", root / "examples/research-foundations.ipynb")
).resolve()


class NotebookDisplay:
    """Render small tables for the exploratory notebook scenarios."""

    @staticmethod
    def show_table(headers: tuple[str, ...], rows: list[tuple[object, ...]]) -> None:
        """Display a small escaped table without adding a dataframe dependency.

        Args:
            headers: Column labels.
            rows: Exact or presentation values to render as text.
        """
        heading = "".join(
            f'<th style="text-align:left;padding:6px">{escape(label)}</th>' for label in headers
        )
        body = "".join(
            "<tr>"
            + "".join(
                f'<td style="padding:6px;vertical-align:top">{escape(str(value))}</td>'
                for value in row
            )
            + "</tr>"
            for row in rows
        )
        display(HTML(f"<table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table>"))

In [2]:
assert platform.python_version() == (root / ".python-version").read_text().strip() == "3.14.7"
installed = Path(alpha_research.__file__).resolve().parent
assert "site-packages" in installed.parts and installed != root / "src/alpha_research"
source_root = root / "src/alpha_research"
runtime_files = sorted(
    path
    for path in source_root.rglob("*")
    if path.is_file() and path.suffix in {".py", ".json", ".typed"}
)
assert runtime_files
runtime_digest = hashlib.sha256()
for source_file in runtime_files:
    relative = source_file.relative_to(source_root)
    source_bytes = source_file.read_bytes()
    assert source_bytes == (installed / relative).read_bytes(), f"Stale installed file: {relative}"
    runtime_digest.update(str(relative).encode() + b"\0" + source_bytes + b"\0")

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip()
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=root, text=True).strip()
working_status = subprocess.check_output(["git", "status", "--short"], cwd=root, text=True).strip()
candidate_diff = subprocess.check_output(
    [
        "git",
        "diff",
        "--binary",
        "HEAD",
        "--",
        "src",
        "configs",
        "pyproject.toml",
        "uv.lock",
        ".python-version",
    ],
    cwd=root,
)
notebook_document = json.loads(notebook_path.read_text())
notebook_source = [
    (cell["cell_type"], "".join(cell["source"])) for cell in notebook_document["cells"]
]
notebook_digest = hashlib.sha256(
    json.dumps(notebook_source, ensure_ascii=False, separators=(",", ":")).encode()
).hexdigest()
identity = {
    "notebook_path": str(notebook_path),
    "attempt_id": os.environ.get("ALPHA_RESEARCH_ATTEMPT_ID", "interactive: unsnapshotted"),
    "commit": commit,
    "branch": branch,
    "working_tree": "changed" if working_status else "clean",
    "python": platform.python_version(),
    "platform": platform.platform(),
    "kernel": sys.executable,
    "package": str(installed),
    "verified_runtime_files": len(runtime_files),
    "runtime_sha256": runtime_digest.hexdigest(),
    "runtime_input_diff_sha256": hashlib.sha256(candidate_diff).hexdigest(),
    "uv_lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest(),
    "okx_config_sha256": hashlib.sha256(
        (root / "configs/history/okx-btc-usdt.json").read_bytes()
    ).hexdigest(),
    "notebook_source_sha256": notebook_digest,
    "notebook_tooling": {
        item.metadata["Name"]: version(item.metadata["Name"])
        for item in distributions()
        if item.metadata["Name"] in {"nbclient", "nbformat", "ipykernel"}
    },
}
NotebookDisplay.show_table(("Identity", "Observed value"), list(identity.items()))

Identity,Observed value
notebook_path,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/examples/research-foundations.ipynb
attempt_id,cac00e61-e5ca-4e0e-8690-037b9462fbbf
commit,972b111b6f0280559e3283e2588a83fb391d1cd3
branch,codex/mr-17-asof-point-events
working_tree,changed
python,3.14.7
platform,macOS-27.0-arm64-arm-64bit-Mach-O
kernel,/Users/lxg/.cache/uv/builds-v0/.tmpEEofYc/bin/python
package,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/.venv/lib/python3.14/site-packages/alpha_research
verified_runtime_files,67


## 2. Which saved samples will I inspect?

Set `ALPHA_OKX_ARCHIVE_DIRECTORY` and `ALPHA_BITFINEX_CAPTURE_DIRECTORY` to the
directories printed by the capture notebook to repeat a particular observation.
Without overrides, this exploration selects the **most recently saved completed
input with the exact maintained configuration**, by manifest save time. That
convenience selection can change after a new acquisition. Pin the printed paths
when a result matters; this is not an accepted research dataset.

The selection table records the paths, fingerprints and original implementation.
The next cells verify the bytes. A selected invalid input fails visibly; it never
triggers a replacement download or fallback to an older sample. No matching input
means run the capture notebook first.


In [3]:
codec = DomainCodec()
okx_config = OKXHistoryConfig.read(root / "configs/history/okx-btc-usdt.json")
bitfinex_config = CaptureConfig.read(root / "configs/live/bitfinex-btc-usd.json")
selected = {}
selection_rows = []
for venue, folder, variable, request_type, config in (
    ("okx", "data/okx/history", "ALPHA_OKX_ARCHIVE_DIRECTORY", OKXArchiveRequest, okx_config),
    (
        "bitfinex",
        "data/bitfinex",
        "ALPHA_BITFINEX_CAPTURE_DIRECTORY",
        CaptureRequest,
        bitfinex_config,
    ),
):
    override = os.environ.get(variable)
    candidates = []
    paths = (
        [Path(override).expanduser()]
        if override
        else [p.parent for p in (root / folder).glob("*/manifest.json")]
    )
    for path in paths:
        request = codec.decode_json((path / "request.json").read_bytes(), request_type)
        if request.config != config:
            if override:
                raise ValueError(f"{variable} differs from the maintained selection")
            continue
        descriptor = codec.decode_json((path / "manifest.json").read_bytes(), RawObject)
        candidates.append((descriptor.ts_ingested, str(path.resolve()), request, descriptor))
    if not candidates:
        raise FileNotFoundError(f"No matching {venue} input; run market-data-capture.ipynb first")
    _, location, request, descriptor = max(candidates, key=lambda item: (item[0], item[1]))
    selected[venue] = (Path(location), request, descriptor)
    selection_rows.extend(
        [
            (
                venue,
                "Selection",
                "explicit directory" if override else "latest matching saved input",
            ),
            (venue, "Directory", location),
            (venue, "Raw SHA-256", descriptor.sha256),
            (venue, "Request identity", request.identity),
            (venue, "Original implementation SHA-256", request.implementation_identity),
        ]
    )
NotebookDisplay.show_table(("Source", "Evidence", "Value"), selection_rows)

Source,Evidence,Value
okx,Selection,explicit directory
okx,Directory,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/data/okx/history/802967bb-05eb-4b7b-9c90-2c964f22e41f
okx,Raw SHA-256,fef59b343d1972c32f67583edc538c5d2f7350e6f3a0059e6f87896d7e8bf676
okx,Request identity,71672efd00a8f3f6c959d0a08f8ce2a8793b3fb84d23f1a3c22b2756a6ec2665
okx,Original implementation SHA-256,22c0e38d88691bd0b1141588260465c471bbde0ff432a6bd1d0dab0505000014
bitfinex,Selection,explicit directory
bitfinex,Directory,/Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/data/bitfinex/a5116c5a-b7d6-4e2b-9ee6-c79839c80f43
bitfinex,Raw SHA-256,a57ddabc5b1283c702ee2c7fd104a2b9cd943e2b41661cfc1d66619b35930163
bitfinex,Request identity,0e1453a2e7e925ad5bc1454a6a7d1e7298eca46e82a0d07b10e7ad8fc1c9ec21
bitfinex,Original implementation SHA-256,22c0e38d88691bd0b1141588260465c471bbde0ff432a6bd1d0dab0505000014


## 3. How much of the historical OKX day did I reconstruct?

The configured date is **8 September 2026**. The reader verifies the original
archive, then reconstructs at most **10,000 records**. The displayed source-time
interval is the processed prefix, not the whole day. L2 combines all reported
size at each price; it supplies no individual order identity or queue position.


In [4]:
okx_directory, okx_request, okx_descriptor = selected["okx"]
history = OKXArchiveStore(okx_directory).replay()
assert history["source_sequence"] is None and history["gaps"] is None
epoch = datetime(1970, 1, 1, tzinfo=UTC)
history_rows = [
    (key, history[key])
    for key in (
        "symbol",
        "representation",
        "records",
        "snapshots",
        "updates",
        "levels",
        "depth",
        "state_available",
        "continuity",
        "stop_reason",
        "entire_archive_replayed",
    )
]
for key in ("first_source_ms", "last_source_ms"):
    value = history[key]
    history_rows.append(
        (
            key.replace("_ms", "_utc"),
            None if value is None else epoch + timedelta(milliseconds=int(value)),
        )
    )
NotebookDisplay.show_table(("Historical observation", "Result"), history_rows)

Historical observation,Result
symbol,BTC-USDT
representation,L2
records,10000
snapshots,1
updates,9999
levels,800
depth,400
state_available,True
continuity,unproven_no_source_sequences
stop_reason,record_limit


**Read the result:** `gaps=None` means continuity is unknown: this archive has no
source sequence numbers. `state_available=True` means a book could be built from
the observed prefix; it does not certify uninterrupted historical coverage.
The archive API currently returns a reconstruction summary, not a research table
of historical prices or features.


## 4. What did the Bitfinex recording actually observe?

This is a saved bounded live recording, not downloadable Bitfinex historical data.
The table reports local receipt times and observed gaps. The R0 subscription
requests depth 25; the final number of retained visible orders is reported
separately. Sequence continuity covers this recording only.


In [5]:
bitfinex_directory, bitfinex_request, bitfinex_descriptor = selected["bitfinex"]
replay = CaptureReplay.read(bitfinex_directory)
summary = replay.summary()
assert isinstance(replay.book, BitfinexBook)
live_rows = [
    (key, summary[key])
    for key in (
        "symbol",
        "messages",
        "snapshots",
        "updates",
        "orders",
        "depth",
        "gaps",
        "synchronized",
        "source_sequence",
        "last_source_time",
    )
]
for key in ("started_ns", "finished_ns"):
    live_rows.append(
        (key.replace("_ns", "_utc"), epoch + timedelta(microseconds=summary[key] // 1000))
    )
NotebookDisplay.show_table(("Recorded observation", "Result"), live_rows)
assert summary["synchronized"] and not summary["gaps"], (
    "This sample is unsuitable for the uninterrupted-book preview; inspect its gaps"
)

Recorded observation,Result
symbol,tBTCUSD
messages,376
snapshots,1
updates,371
orders,50
depth,25
gaps,[]
synchronized,True
source_sequence,373
last_source_time,None


## 5. What visible prices and sizes can I inspect?

Read a few individual Bitfinex orders nearest each side's best price. Prices are
**USD per BTC**, sizes are **BTC**, and the displayed spread is **USD per BTC**.
Decimal values remain exact and are rendered as text. The spread is a descriptive
observation, not a trading return; it excludes fees, impact and fill uncertainty.


In [6]:
preview_rows = []
best = {}
for side in ("buy", "sell"):
    orders = sorted(
        (order for order in replay.book.orders.values() if order.side == side),
        key=lambda order: order.price,
        reverse=side == "buy",
    )
    assert orders, f"No observed {side} side"
    best[side] = orders[0].price
    for order in orders[:3]:
        assert isinstance(order.price, Decimal) and isinstance(order.size, Decimal)
        preview_rows.append((side, order.order_id, str(order.price), str(order.size)))
NotebookDisplay.show_table(("Side", "Visible order ID", "USD / BTC", "BTC"), preview_rows)
NotebookDisplay.show_table(
    ("Measure", "USD / BTC"),
    [
        ("Best bid", best["buy"]),
        ("Best ask", best["sell"]),
        ("Observed spread", best["sell"] - best["buy"]),
    ],
)

Side,Visible order ID,USD / BTC,BTC
buy,244232632883,75645,0.00180442
buy,244230114777,75639,0.0002
buy,244218842321,75638,0.0002
sell,244216587539,75655,0.00022
sell,244232116576,75660,0.03522185
sell,244217678512,75660,0.00022


Measure,USD / BTC
Best bid,75645
Best ask,75655
Observed spread,10


## 6. When did the market event happen, and when did I know about it?

OKX historical records have source timestamps, but their download time is much
later and there is no historical receipt clock. Bitfinex R0 has local receipt order
and sequences, but no exchange timestamp in these book messages. Neither clock
can fill in the other's missing information. UTC display truncates nanoseconds to
microseconds; original Bitfinex nanoseconds remain in the journal.


In [7]:
NotebookDisplay.show_table(
    ("Sample", "Source clock", "Local receipt evidence", "Saved bytes"),
    [
        (
            "OKX historical BTC-USDT",
            "Unix milliseconds in records; continuity unknown",
            f"Archive received {history['download_received']}; per-record receipt absent",
            history["saved_bytes"],
        ),
        (
            "Bitfinex recorded BTC/USD",
            "Absent in R0 messages",
            f"Journal nanoseconds {summary['started_ns']} to {summary['finished_ns']}",
            bitfinex_descriptor.size_bytes,
        ),
    ],
)

Sample,Source clock,Local receipt evidence,Saved bytes
OKX historical BTC-USDT,Unix milliseconds in records; continuity unknown,Archive received 2026-09-16T08:49:14.475436+00:00; per-record receipt absent,109423187
Bitfinex recorded BTC/USD,Absent in R0 messages,Journal nanoseconds 1789548508888411000 to 1789548529076738000,52908


## 7. What prevents me from analysing damaged or disconnected input?

These **controlled failures** modify temporary copies only: remove saved input,
change a saved byte, alter archive discovery, and inject a disconnect into a
separate reconstruction. The original recordings stay intact. Inspect the explicit
rejections and the unavailable book after the gap. A fresh subscription/snapshot
is needed for recovery; this example does not fill the gap with stale quotes.


In [8]:
# standard library imports
import shutil
from tempfile import TemporaryDirectory

failure_rows = []
with TemporaryDirectory(prefix="orderbook-input-checks-") as temporary:
    copy = Path(temporary)
    for name in ("request.json", "manifest.json"):
        shutil.copyfile(bitfinex_directory / name, copy / name)
    raw = copy / bitfinex_descriptor.sha256
    for label, expected in (
        ("Missing Bitfinex bytes", FileNotFoundError),
        ("Changed Bitfinex bytes", ValueError),
    ):
        if expected is ValueError:
            content = (bitfinex_directory / bitfinex_descriptor.sha256).read_bytes()
            raw.write_bytes(bytes([content[0] ^ 1]) + content[1:])
        try:
            CaptureReplay.read(copy)
        except expected as error:
            failure_rows.append((label, type(error).__name__))
        else:
            raise AssertionError(f"Accepted {label}")
    shutil.copyfile(okx_directory / "request.json", copy / "request.json")
    (copy / "index.json").write_bytes(b"{}")
    try:
        OKXArchiveStore(copy).replay()
    except ValueError as error:
        failure_rows.append(("Changed OKX discovery", str(error)))
    else:
        raise AssertionError("Accepted changed OKX discovery")

exercise = CaptureReplay(bitfinex_config)
for record in CaptureJournal(bitfinex_directory, bitfinex_request).records():
    if record.kind == "end":
        break
    exercise.apply(record)
exercise.apply(
    CaptureRecord(
        kind="gap",
        epoch=exercise.epoch,
        received_ns=exercise.finished_ns,
        reason="synthetic_disconnect",
    )
)
assert not exercise.book.ready
failure_rows.append(
    ("Injected disconnect", f"Book usable: {exercise.book.ready}; {exercise.gaps[-1]['reason']}")
)
NotebookDisplay.show_table(("Controlled case", "Observed result"), failure_rows)

Controlled case,Observed result
Missing Bitfinex bytes,FileNotFoundError
Changed Bitfinex bytes,ValueError
Changed OKX discovery,saved OKX archive discovery differs from request
Injected disconnect,Book usable: False; synthetic_disconnect


## 8. What can I conclude, and what should I investigate next?

The last check reopens the Bitfinex input and compares the entire sequence of
reconstructed states. This establishes repeatability from the same saved bytes
and installed code, not data completeness or executable fills.


In [9]:
again = CaptureReplay.read(bitfinex_directory)
assert again.summary()["trace_identity"] == summary["trace_identity"]
NotebookDisplay.show_table(
    ("Research finding", "Evidence or limit"),
    [
        (
            "Historical liquidity observations",
            f"OKX: {history['records']} prefix records, {history['levels']} final levels",
        ),
        (
            "Recorded visible orders",
            f"Bitfinex: {summary['messages']} messages, {summary['orders']} final orders",
        ),
        ("Repeatability", f"Matching Bitfinex state trace: {summary['trace_identity']}"),
        ("Controlled failures", f"{len(failure_rows)} explicit outcomes"),
        (
            "Not established",
            "Same-time prices, full coverage, historical eligibility, queue priority or fills",
        ),
        (
            "Next roadmap MR",
            "MR 10: admit economic duplicates and corrections explicitly; still planned",
        ),
    ],
)

Research finding,Evidence or limit
Historical liquidity observations,"OKX: 10000 prefix records, 800 final levels"
Recorded visible orders,"Bitfinex: 376 messages, 50 final orders"
Repeatability,Matching Bitfinex state trace: 283533b937763c40d108517fc22ddbbc2d10a77820a8de498cc631f37c11d375
Controlled failures,4 explicit outcomes
Not established,"Same-time prices, full coverage, historical eligibility, queue priority or fills"
Next roadmap MR,MR 10: admit economic duplicates and corrections explicitly; still planned


For a new observation, deliberately run the [capture notebook](market-data-capture.ipynb)
and pin its printed directories before rerunning this study. For timing, exact
financial units and saved-row boundary exercises, use the independent
[offline reliability lab](foundation-contract-checks.ipynb). Keep an influencing
trial in the research journal; these local snapshots are not its backup.


## 9. Can two observations have sequence zero without being the same occurrence? (NB 09)

**Sample and assumptions:** the following tiny Binance individual-trade archives
are synthetic teaching inputs, not new market observations. Two versions of a BTC
file start with the same trade; another file belongs to ETH. We resolve the existing
configuration, publish exact raw bytes locally, then use the installed dataset
pipeline to decode them. No network, revision policy or dataset publication is involved.

Read the table from the input binding and raw digest to source position, then to
occurrence identity. A repeated build must reproduce these identities. Trade IDs
are economic identifiers with separate duplicate/correction rules demonstrated in section 10.

In [10]:
# standard library imports
from zipfile import ZIP_DEFLATED, ZipFile, ZipInfo

# third party library imports
from msgspec.structs import replace

# project library imports
from alpha_research.configuration import Configuration
from alpha_research.data.events import Disconnect
from alpha_research.data.manifests import AcquisitionManifest
from alpha_research.data.normalization import Occurrence
from alpha_research.data.providers.binance.trades import ArchiveTrade
from alpha_research.data.providers.tardis.decoder import TardisDecoder
from alpha_research.data.raw_store import RawStore
from alpha_research.data.requests import SourceRequest
from alpha_research.data.venues.binance.payloads import BinancePayloadDecoder
from alpha_research.pipelines.dataset import iter_occurrences
from alpha_research.reference_data import CatalogSnapshot

archive_config = Configuration.read(root / "configs/data-jobs/binance-archive.json")
archive_catalog = codec.decode_json(
    (root / "configs/reference/binance-archive-spot.json").read_bytes(), CatalogSnapshot
)
archive_run = archive_config.resolve("btc-archive", archive_catalog)
archive_run = replace(archive_run, instruments=archive_run.catalog.instruments)
archive_requests = SourceRequest.partition(archive_run)
mr09_first = b"12,100.00,2.0,199.999,1717200000001,true,true\r\n"
mr09_second = b"11,101.00,3.0,303.000,1717200000001,false,true\n"
mr09_items = []
mr09_failures = []
with TemporaryDirectory(prefix="mr09-occurrences-") as temporary:
    local = Path(temporary)
    store = RawStore(local)
    descriptors = []
    for binding, content in zip(
        archive_requests, (mr09_first + mr09_second, mr09_first), strict=True
    ):
        path = local / "source.zip"
        member = ZipInfo(f"{binding.instrument.symbol}-trades-{binding.start.date()}.csv")
        member.compress_type = ZIP_DEFLATED
        with ZipFile(path, "w") as archive:
            archive.writestr(member, content)
        descriptor = RawObject.from_file(
            path,
            request=binding,
            retrieval_reference="synthetic:nb09",
            ts_received=datetime(2026, 9, 16, tzinfo=UTC),
            ts_ingested=datetime(2026, 9, 16, tzinfo=UTC),
        )
        store.publish(path, descriptor=descriptor, request=binding)
        descriptors.append(descriptor)
    state = AcquisitionManifest(run=archive_run, completed=tuple(descriptors))
    state.save(local / "manifest.json")
    restored = AcquisitionManifest.read(local / "manifest.json", run=archive_run, store=store)
    mr09_items = list(iter_occurrences(restored, store, max_bytes=4096))
    assert list(iter_occurrences(restored, store, max_bytes=4096)) == mr09_items
    assert [item.position for item in mr09_items] == [0, 1, 0]
    assert len({item.identity for item in mr09_items}) == 3
    # An additional file version starts with the same BTC trade and sequence zero.
    with ZipFile(path, "w", compression=ZIP_DEFLATED) as archive:
        archive.writestr("BTCUSDT-trades-2024-06-01.csv", mr09_first)
    original_run = replace(archive_run, instruments=(archive_run.instruments[0],))
    original_request = SourceRequest.partition(original_run)[0]
    original = RawObject.from_file(
        path,
        request=original_request,
        retrieval_reference="synthetic:nb09-original",
        ts_received=descriptors[0].ts_received,
        ts_ingested=descriptors[0].ts_ingested,
    )
    store.publish(path, descriptor=original, request=original_request)
    previous = next(
        iter_occurrences(
            AcquisitionManifest(run=original_run, completed=(original,)), store, max_bytes=4096
        )
    )
    assert previous.position == mr09_items[0].position == 0
    assert previous.event == mr09_items[0].event
    assert previous.identity != mr09_items[0].identity
    mr09_items.append(previous)
    try:
        replace(previous, event=mr09_items[2].event)
    except ValueError as error:
        mr09_failures.append(("Unselected instrument", str(error)))
    else:
        raise AssertionError("foreign event accepted")
    try:
        replace(previous, request=archive_requests[1])
    except ValueError as error:
        mr09_failures.append(("Incompatible raw binding", str(error)))
    else:
        raise AssertionError("foreign raw binding accepted")

NotebookDisplay.show_table(
    ("Binding", "Raw digest", "Position", "Trade ID", "Occurrence identity"),
    [
        (
            item.request.instrument.symbol,
            item.origin.sha256,
            item.position,
            item.event.trade_id,
            item.identity,
        )
        for item in mr09_items
    ],
)

Binding,Raw digest,Position,Trade ID,Occurrence identity
BTCUSDT,f2afa9ba9084181772ee567881d4a3fbad26b5a658fee580a374c9cb3fe5c600,0,12,a6c112ecf15ba80e70f1821cdf65905a581a06b70818e3de2720ded634532d04
BTCUSDT,f2afa9ba9084181772ee567881d4a3fbad26b5a658fee580a374c9cb3fe5c600,1,11,52ad234b26653d06a6017e7488f646813efa7c5ed448cc266c6c14f06f3810cd
ETHUSDT,633cbfce42192b841fdc133b3c865165c5b1f958928287f07b315747561dba7d,0,12,cbc806844d13f3e5e9d797e53de87069be0822cd9f4f6811a1ece8ac57e16736
BTCUSDT,1b62fd3c07cb9d29670617192c64f63cceadf4a5277c5f2114bbc92cab1682e9,0,12,fd1fbdb0acb5caef4a11c94f096062647b7ea4b23e92786fab609f488196bcb9


**Observed output:** all four occurrences remain attributable, including both
positions zero in the BTC file versions and the independent ETH binding. The two
rows at equal exchange time retain their file order (trade IDs 12 then 11).

The next table retains the publisher's integer milliseconds and decimal scales.
The synthetic quote quantity deliberately differs from price × quantity; decoding
preserves it for later validation. Download receipt belongs to the raw object.
Capture and historical knowledge are absent. No Tardis timestamp is fabricated.

In [11]:
archive_event = mr09_items[0].event
assert isinstance(archive_event, ArchiveTrade)
assert archive_event.ts_source_received is archive_event.ts_knowledge is None
assert archive_event.quote_quantity == Decimal("199.999")
assert codec.decode_json(codec.encode_json(mr09_items[0]), Occurrence) == mr09_items[0]
NotebookDisplay.show_table(
    ("Field", "Value"),
    [
        ("Source integer / unit", (archive_event.timestamp, archive_event.timestamp_unit)),
        ("UTC event time", archive_event.ts_event),
        (
            "Price / base quantity / quote quantity",
            (
                str(archive_event.price),
                str(archive_event.quantity),
                str(archive_event.quote_quantity),
            ),
        ),
        (
            "Capture / historical knowledge",
            (archive_event.ts_source_received, archive_event.ts_knowledge),
        ),
        ("Raw acquisition receipt", mr09_items[0].origin.ts_received),
        ("Original CSV including terminator", repr(mr09_items[0].raw)),
    ],
)

Field,Value
Source integer / unit,"(1717200000001, 'ms')"
UTC event time,2024-06-01 00:00:00.001000+00:00
Price / base quantity / quote quantity,"('100.00', '2.0', '199.999')"
Capture / historical knowledge,"(None, None)"
Raw acquisition receipt,2026-09-16 00:00:00+00:00
Original CSV including terminator,"b'12,100.00,2.0,199.999,1717200000001,true,true\r\n'"


### Which input does an instrument-less disconnect affect?

Use the unchanged Tardis decoder on identical synthetic blank files for two
independent bindings. Both controls say `instrument_id=None` and `stream=global`,
but their occurrence envelopes retain BTC and ETH respectively. Their raw digests
and legacy event IDs can match; their occurrence identities must differ. These
saved-file subscriptions have no live session identifier. Later query/replay work
must retain the binding rather than applying a control to every repository input.

In [12]:
fixture_config = Configuration.read(root / "configs/data-jobs/fixture-inspection.json")
fixture_catalog = codec.decode_json(
    (root / "configs/reference/binance-spot.json").read_bytes(), CatalogSnapshot
)
captured_run = fixture_config.resolve("both-fixture", fixture_catalog)
controls = []
with TemporaryDirectory(prefix="mr09-controls-") as temporary:
    path = Path(temporary) / "blank.txt"
    path.write_bytes(b"\r\n")
    for binding in SourceRequest.partition(captured_run)[:2]:
        origin = RawObject.from_file(
            path,
            request=binding,
            retrieval_reference="synthetic:nb09-control",
            ts_received=datetime(2026, 9, 16, tzinfo=UTC),
            ts_ingested=datetime(2026, 9, 16, tzinfo=UTC),
        )
        control = TardisDecoder(
            BinancePayloadDecoder(captured_run.instruments), binding.instrument.source
        ).decode_line(
            "",
            source_sequence=0,
            local_received=origin.ts_received,
            local_ingested=origin.ts_ingested,
            source_digest=origin.sha256,
            disconnect_capture_time=binding.end,
            disconnect_capture_reason="interval_end",
        )
        assert isinstance(control, Disconnect)
        controls.append(
            Occurrence(
                request=binding, origin=origin, position=0, raw=path.read_bytes(), event=control
            )
        )
assert controls[0].origin.sha256 == controls[1].origin.sha256
assert controls[0].event == controls[1].event
assert controls[0].identity != controls[1].identity
NotebookDisplay.show_table(
    ("Affected binding", "Instrument / stream on control", "Raw digest", "Occurrence identity"),
    [
        (
            item.request.instrument.symbol,
            (item.event.instrument_id, item.event.source_stream),
            item.origin.sha256,
            item.identity,
        )
        for item in controls
    ],
)
NotebookDisplay.show_table(("Required failure", "Observed rejection"), mr09_failures)

Affected binding,Instrument / stream on control,Raw digest,Occurrence identity
BTCUSDT,"(None, 'global')",7eb70257593da06f682a3ddda54a9d260d4fc514f645237f5ca74b08f8da61a6,e4d819d97b4b7851380be1452500cabd51a5c9feca49c05ca30986b664455dde
ETHUSDT,"(None, 'global')",7eb70257593da06f682a3ddda54a9d260d4fc514f645237f5ca74b08f8da61a6,1292cc0b8438c68fd9f3cb581073f0e27f33de3a47892f04f8f06479968723d4


Required failure,Observed rejection
Unselected instrument,occurrence event instrument differs from selected binding
Incompatible raw binding,occurrence raw origin differs from request binding


**Interpretation and next question:** the installed pipeline can trace decoded
archive trades to exact saved bytes, preserve order and rebuild stable occurrence
identities. The controlled failures reject unselected instruments and incompatible
raw bindings. The retained tests additionally reject malformed/nonfinite amounts,
wrong timestamp units/days, missing or mutated objects and unsafe records.

This is MR 09 occurrence normalization. It does not merge duplicates, order
corrections, establish historical knowledge/eligibility/coverage, or publish an
accepted dataset. The earlier real book summaries remain separate observations.
**Next, section 10 (NB 10):** determine which occurrences represent the same
economic trade and how correction evidence preserves earlier versions.

## 10. Does a later correction change what I could have known earlier? (NB 10)

**Sample and assumptions:** three tiny synthetic Binance archives contain an
original trade, a copy shifted by an inserted row, and a changed price. Their
publication/availability evidence below is deliberately authored and labeled
synthetic. It is not historical Binance evidence. The installed dataset pipeline
reads the saved bytes and admits one economic trade per source/venue/instrument/
stream/trade ID. Raw positions identify observations, not economic versions.

Inspect the original's two raw origins, the appended correction, and the original's
unknown knowledge time. An unevidenced conflict must fail before any result returns.


In [13]:
# project library imports
from alpha_research.data.revisions import Correction, RevisionConflictError, TradeRevision
from alpha_research.pipelines.dataset import admit_saved_trades

mr10_failures = []
with TemporaryDirectory(prefix="mr10-revisions-") as temporary:
    local = Path(temporary)
    store = RawStore(local)
    states = []
    for content in (
        mr09_first,
        mr09_second + mr09_first,
        mr09_second + mr09_first.replace(b"100.00", b"102.00"),
    ):
        path = local / "source.zip"
        member = ZipInfo("BTCUSDT-trades-2024-06-01.csv")
        member.compress_type = ZIP_DEFLATED
        with ZipFile(path, "w") as archive:
            archive.writestr(member, content)
        origin = RawObject.from_file(
            path,
            request=original_request,
            retrieval_reference="synthetic:nb10",
            ts_received=datetime(2026, 9, 16, tzinfo=UTC),
            ts_ingested=datetime(2026, 9, 16, tzinfo=UTC),
        )
        store.publish(path, descriptor=origin, request=original_request)
        states.append(AcquisitionManifest(run=original_run, completed=(origin,)))
    duplicates = admit_saved_trades(
        tuple(states[:2]), store, max_bytes=4096, policy="reject", corrections=()
    )
    archive_original = next(head for head in duplicates if head.key[-1] == 12)
    assert len(duplicates) == 2 and len(archive_original.occurrences) == 2
    assert {item.position for item in archive_original.occurrences} == {0, 1}
    assert archive_original.ts_knowledge is None
    revised = tuple(iter_occurrences(states[-1], store, max_bytes=4096))[1]
    archive_evidence = Correction(
        supersedes=archive_original.identity,
        replacement=TradeRevision(occurrences=(revised,)).identity,
        ts_published=datetime(2024, 6, 2, tzinfo=UTC),
        ts_available=datetime(2024, 6, 2, 0, 0, 1, tzinfo=UTC),
        evidence_reference="synthetic:nb10-authored-correction",
    )
    try:
        admit_saved_trades(tuple(states), store, max_bytes=4096, policy="evidenced", corrections=())
    except RevisionConflictError as error:
        mr10_failures.append(("Conflict without evidence", str(error)))
        assert len(error.revisions) == 3
    else:
        raise AssertionError("unevidenced conflict admitted")
    admitted = admit_saved_trades(
        tuple(states), store, max_bytes=4096, policy="evidenced", corrections=(archive_evidence,)
    )
    archive_corrected = next(head for head in admitted if head.key[-1] == 12)
    assert archive_corrected.previous == archive_original
    assert archive_corrected.ts_knowledge == archive_evidence.ts_available
    assert (
        codec.decode_json(codec.encode_json(archive_corrected), TradeRevision) == archive_corrected
    )

NotebookDisplay.show_table(
    ("Version", "Price", "Economic payload identity", "Raw positions", "Effective knowledge"),
    [
        (
            label,
            version.occurrences[0].event.price,
            version.identity,
            [(item.origin.sha256[:12], item.position) for item in version.occurrences],
            version.ts_knowledge,
        )
        for label, version in (
            ("Original + duplicate", archive_original),
            ("Correction", archive_corrected),
        )
    ],
)

Version,Price,Economic payload identity,Raw positions,Effective knowledge
Original + duplicate,100.00,fcf81aaf5c297bdcd4b6acc78dcb9704d3e47e4f0e962ea5bfd0a5435f6da0d9,"[('3fc08308bd24', 1), ('92a3db322729', 0)]",None
Correction,102.00,06d4f503e5f06bc6965ef5d7928daacce088114d314a1743616b6cb5e864e4ef,"[('b8d303d28518', 1)]",2024-06-02 00:00:01+00:00


**Observed output:** the shifted duplicate contributes once and retains both raw
origins. The changed payload has an explicit link to its predecessor. The original
archive remains historically unavailable (`None`); only the synthetic correction
assertion establishes the later version's knowledge. Raw acquisition in 2026 does
not fill in missing 2024 clocks. Publisher quote amounts are retained unchanged;
reconciliation/quality belongs to MR 13.

### Reconstruct the earlier price when an original capture is known

Use the existing synthetic Tardis fixture with its original submicrosecond capture
text, then supply a price correction published at 00:00:09 and available at
00:00:10. The correction retains the original event/capture clocks. Walk the saved
version envelopes at three decision times: before capture, before correction,
and exactly at correction availability. This is a small notebook inspection;
the shared query reader remains later roadmap work.


In [14]:
# project library imports
from alpha_research.data.venues.binance.events import Trade

captured_binding = SourceRequest.partition(captured_run)[0]
original_line = (root / "tests/fixtures/m0/btcusdt.txt").read_bytes()
captured_versions = []
with TemporaryDirectory(prefix="mr10-captured-") as temporary:
    path = Path(temporary) / "trade.txt"
    for raw in (
        original_line,
        original_line.replace(b'"p":"67540.01000000"', b'"p":"67541.01000000"'),
    ):
        path.write_bytes(raw)
        origin = RawObject.from_file(
            path,
            request=captured_binding,
            retrieval_reference="synthetic:nb10-capture",
            ts_received=datetime(2026, 9, 16, tzinfo=UTC),
            ts_ingested=datetime(2026, 9, 16, tzinfo=UTC),
        )
        event = TardisDecoder(
            BinancePayloadDecoder(captured_run.instruments), captured_binding.instrument.source
        ).decode_line(
            raw.decode().strip(),
            source_sequence=0,
            local_received=origin.ts_received,
            local_ingested=origin.ts_ingested,
            source_digest=origin.sha256,
        )
        assert isinstance(event, Trade)
        captured_versions.append(
            Occurrence(request=captured_binding, origin=origin, position=0, raw=raw, event=event)
        )
original_version = TradeRevision(occurrences=(captured_versions[0],))
correction_evidence = Correction(
    supersedes=original_version.identity,
    replacement=TradeRevision(occurrences=(captured_versions[1],)).identity,
    ts_published=datetime(2024, 6, 1, 0, 0, 9, tzinfo=UTC),
    ts_available=datetime(2024, 6, 1, 0, 0, 10, tzinfo=UTC),
    evidence_reference="synthetic:nb10-late-correction",
)
current_version = TradeRevision.admit(
    captured_versions, policy="evidenced", corrections=(correction_evidence,)
)[0]
assert current_version.previous == original_version
assert original_version.ts_knowledge == datetime(2024, 6, 1, 0, 0, 0, 4103, tzinfo=UTC)
assert current_version.ts_knowledge == correction_evidence.ts_available
# A notebook-local inspection of retained envelopes, not a packaged as-of reader.
decision_rows = []
for decision in (
    datetime(2024, 6, 1, tzinfo=UTC),
    datetime(2024, 6, 1, 0, 0, 1, tzinfo=UTC),
    datetime(2024, 6, 1, 0, 0, 10, tzinfo=UTC),
):
    visible = current_version
    while visible.previous is not None and visible.ts_knowledge > decision:
        visible = visible.previous
    available = visible.ts_knowledge is not None and visible.ts_knowledge <= decision
    decision_rows.append(
        (
            decision,
            visible.occurrences[0].event.price if available else "Unavailable",
            visible.ts_knowledge if available else None,
        )
    )
assert [row[1] for row in decision_rows] == [
    "Unavailable",
    Decimal("67540.01000000"),
    Decimal("67541.01000000"),
]
NotebookDisplay.show_table(
    ("Decision time", "Reconstructible price", "Effective knowledge"), decision_rows
)

Decision time,Reconstructible price,Effective knowledge
2024-06-01 00:00:00+00:00,Unavailable,None
2024-06-01 00:00:01+00:00,67540.01000000,2024-06-01 00:00:00.004103+00:00
2024-06-01 00:00:10+00:00,67541.01000000,2024-06-01 00:00:10+00:00


**Observed output:** no price is usable before capture. The old price remains
reconstructible at 00:00:01 even after the correction has arrived. At the inclusive
00:00:10 availability boundary the corrected price becomes usable. The earlier
version and raw bytes remain unchanged.

### Which claims must admission refuse?

Try absent evidence, inconsistent/naive clocks and an instrument-less control.
Controls retain their MR 09 binding scope and are not economic trades.


In [15]:
for label, fields in (
    ("Missing evidence reference", {"evidence_reference": ""}),
    ("Availability before publication", {"ts_available": datetime(2024, 6, 1, tzinfo=UTC)}),
    ("Naive publication time", {"ts_published": datetime(2024, 6, 1, 0, 0, 9)}),
):
    try:
        replace(correction_evidence, **fields)
    except ValueError as error:
        mr10_failures.append((label, str(error)))
    else:
        raise AssertionError(f"invalid evidence accepted: {label}")
try:
    TradeRevision.admit(controls, policy="reject", corrections=())
except TypeError as error:
    mr10_failures.append(("Instrument-less control is not a trade", str(error)))
else:
    raise AssertionError("control deduplicated as a trade")
NotebookDisplay.show_table(("Required failure", "Observed rejection"), mr10_failures)

Required failure,Observed rejection
Conflict without evidence,"unresolved trade revisions: fcf81aaf5c297bdcd4b6acc78dcb9704d3e47e4f0e962ea5bfd0a5435f6da0d9, 139faa5f2a9d7b7bfbcc12aab385ccfa62e00e8853120ae95c8cde54a92b90c1, 06d4f503e5f06bc6965ef5d7928daacce088114d314a1743616b6cb5e864e4ef"
Missing evidence reference,correction requires a printable trimmed evidence reference
Availability before publication,correction must change payload and be available after publication
Naive publication time,correction clocks must be timezone-aware UTC
Instrument-less control is not a trade,revision admission supports individual trades only


**Interpretation and limits:** MR 10 admits exact economic duplicates with all
lineage and appends explicitly evidenced corrections. Unknown, branched, cyclic,
dangling or temporally reversed correction links fail; originals stay available
for reconstruction. The batch is held in memory and correction chains require
distinct economic payloads. A later reversion to an identical payload requires
additional source revision evidence and is rejected by this bounded policy.

These examples establish software behavior, not empirical historical revision
availability, completeness, quality, publication or a P1 result. No downloader,
source configuration, stored raw format or frozen experiment changed.
See the [MR 10 evidence record](../docs/roadmap/mr-10-implementation.md).

**Next question — MR 11:** what evidence establishes historical eligibility for
the selected instruments, and when was it known? The next section reads these assertions through MR 11.


## 11. Does a historical validity date prove that membership was known then? (NB 11)

**Sample and assumptions:** read the retained [synthetic eligibility file](../tests/fixtures/mr11/eligibility.json)
against the unchanged archive catalog. It pairs a **2026-authored assumption** with
an assertion whose explicitly **synthetic evidence models availability in 2024**.
The latter exercises an evidenced-in-sample clock; it is not an authenticated
historical Binance announcement and does not resolve P1's missing evidence.

**Action:** compare valid time, knowledge, evidence basis and retained identity.
Inspect the excluded ETH member after a hypothetical delisting. Its exclusion
extends beyond catalog validity: preserving it prevents a survivor-only history.
No decision-time universe selection is implemented in this checkpoint.


In [16]:
# project library imports
from alpha_research.data.eligibility import EligibilityAssertions

mr11_path = root / "tests/fixtures/mr11/eligibility.json"
mr11_catalog = codec.decode_json(
    (root / "configs/reference/binance-archive-spot.json").read_bytes(), CatalogSnapshot
)
mr11_catalog_bytes = codec.encode_json(mr11_catalog)
mr11_history = EligibilityAssertions.read(mr11_path, catalog=mr11_catalog)
assumption, evidenced, included, excluded = mr11_history.assertions
assert assumption.valid_from == evidenced.valid_from == datetime(2024, 6, 1, tzinfo=UTC)
assert assumption.ts_knowledge == datetime(2026, 9, 16, tzinfo=UTC)
assert evidenced.ts_knowledge == datetime(2024, 6, 1, tzinfo=UTC)
assert evidenced.evidence.basis == "synthetic"
assert included.instrument_id == excluded.instrument_id and excluded.state == "excluded"
assert excluded.valid_to > mr11_catalog.instruments[1].valid_to
assert codec.encode_json(mr11_history.catalog) == mr11_catalog_bytes
assert codec.decode_json(codec.encode_json(mr11_history), EligibilityAssertions) == mr11_history
NotebookDisplay.show_table(
    (
        "Member",
        "State",
        "Valid [from, to)",
        "Knowledge",
        "Evidence basis",
        "Evidence identity",
        "Reason",
    ),
    [
        (
            next(
                item.symbol
                for item in mr11_catalog.instruments
                if item.instrument_id == row.instrument_id
            ),
            row.state,
            (row.valid_from, row.valid_to),
            row.ts_knowledge,
            row.evidence.basis,
            row.evidence.identity,
            row.reason,
        )
        for row in mr11_history.assertions
    ],
)

Member,State,"Valid [from, to)",Knowledge,Evidence basis,Evidence identity,Reason
BTCUSDT,included,"(datetime.datetime(2024, 6, 1, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2024, 6, 2, 0, 0, tzinfo=datetime.timezone.utc))",2026-09-16 00:00:00+00:00,research_assumption,synthetic:mr11:authored-assumption-v1,Authored 2026 research assumption; no historical membership proof.
BTCUSDT,included,"(datetime.datetime(2024, 6, 1, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2024, 6, 2, 0, 0, tzinfo=datetime.timezone.utc))",2024-06-01 00:00:00+00:00,synthetic,synthetic:mr11:sample-announcement-v1,Synthetic evidence models membership known at the start of the sample.
ETHUSDT,included,"(datetime.datetime(2024, 6, 1, 0, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2024, 6, 1, 12, 0, tzinfo=datetime.timezone.utc))",2024-06-01 00:00:00+00:00,synthetic,synthetic:mr11:sample-announcement-v1,Synthetic member retained before its hypothetical delisting.
ETHUSDT,excluded,"(datetime.datetime(2024, 6, 1, 12, 0, tzinfo=datetime.timezone.utc), datetime.datetime(2024, 6, 3, 0, 0, tzinfo=datetime.timezone.utc))",2024-06-01 00:00:00+00:00,synthetic,synthetic:mr11:sample-announcement-v1,Synthetic delisting; retain the excluded member beyond catalog validity.


**Observed output:** both BTC assertions describe the same 2024 valid interval.
The authored assumption becomes known only in 2026; the synthetic source evidence
models knowledge at the sample start. The catalog still says `research_assumption`
and retains its exact original currencies, ticks and authored date.

**Required failures:** remove evidence, backdate knowledge, backdate the assumption's
own availability, introduce a contradictory overlap and change selected catalog units.
Then reopen the original file to demonstrate recovery without partial state.


In [17]:
mr11_failures = []
with TemporaryDirectory(prefix="nb11-eligibility-") as temporary:
    path = Path(temporary) / "eligibility.json"
    raw = mr11_path.read_text()
    for label, payload in (
        (
            "Missing evidence identity",
            raw.replace('"identity": "synthetic:mr11:authored-assumption-v1"', '"identity": ""'),
        ),
        (
            "Invented historical knowledge",
            raw.replace(
                '"ts_knowledge": "2026-09-16T00:00:00Z"', '"ts_knowledge": "2024-06-01T00:00:00Z"'
            ),
        ),
        (
            "Backdated authored availability",
            raw.replace(
                '"available_at": "2026-09-16T00:00:00Z"', '"available_at": "2024-06-01T00:00:00Z"'
            ),
        ),
        (
            "Contradictory overlapping assertion",
            raw.replace('"state": "included"', '"state": "excluded"', 1),
        ),
    ):
        path.write_text(payload)
        try:
            EligibilityAssertions.read(path, catalog=mr11_catalog)
        except ValueError as error:
            mr11_failures.append((label, str(error)))
        else:
            raise AssertionError(f"invalid eligibility accepted: {label}")
    first, second = mr11_catalog.instruments
    changed_catalog = replace(
        mr11_catalog, instruments=(replace(first, price_tick=Decimal("0.02")), second)
    )
    try:
        EligibilityAssertions.read(mr11_path, catalog=changed_catalog)
    except ValueError as error:
        mr11_failures.append(("Changed catalog units", str(error)))
    else:
        raise AssertionError("changed catalog admitted")
    path.write_text(raw)
    assert EligibilityAssertions.read(path, catalog=mr11_catalog) == mr11_history
assert len(mr11_failures) == 5
NotebookDisplay.show_table(("Required failure", "Observed rejection"), mr11_failures)

Required failure,Observed rejection
Missing evidence identity,evidence identity must be nonempty without surrounding whitespace - at `$.assertions[0].evidence`
Invented historical knowledge,eligibility knowledge cannot precede evidenced availability - at `$.assertions[0]`
Backdated authored availability,research assumption availability must equal its authored recording - at `$.assertions[0].evidence`
Contradictory overlapping assertion,contradictory eligibility overlap: 11111111-1111-4111-8111-111111111111
Changed catalog units,eligibility file differs from the selected catalog/units


**Interpretation and limits:** strict file reading preserves four assertions, including
the delisted member, and rejects all five invalid cases. The evidence identity is a
retained document/revision reference supplied by the author; the reader validates
its metadata and temporal consistency but does not fetch or authenticate the document.
Synthetic rows establish software behavior only. A 2026 catalog cannot establish
historical membership or historical unit/filter truth, and P1 evidence remains unresolved.

**Next research question:** which retained assertion applies at a decision, using
validity and knowledge independently? Section 12 now exercises **MR 12**, including explicit unknown
membership and append-only corrections.


## 12. Which universe could a decision actually have used? (NB 12)

**Question:** can a later listing correction change the universe reconstructed for
an earlier decision? **Sample:** the [MR 12 synthetic fixture](../tests/fixtures/mr12/eligibility.json)
retains one original BTC inclusion and the two ETH assertions from NB 11. It appends
an explicitly linked BTC correction learned at noon: the listing's start moves to
02:00, and the corrected interval is excluded. The separate 2026 BTC assumption is
not part of this scenario. No source document is authenticated by this exercise.

**Action:** query independent `valid_at` and inclusive `known_at` clocks one microsecond
before, at and after each boundary. A correction retires the **whole** predecessor
when known; a removed valid interval becomes unknown, never a fallback inclusion.

In [18]:
# project library imports
from alpha_research.data.eligibility import EligibilityAssertion
from alpha_research.instrument_identity import InstrumentId

mr12_history = EligibilityAssertions.read(
    root / "tests/fixtures/mr12/eligibility.json", catalog=mr11_catalog
)
mr12_original, _, mr12_delisted, mr12_correction = mr12_history.assertions
mr12_before = replace(mr12_history, schema_version="1", assertions=mr12_history.assertions[:-1])
assert mr12_before.append(mr12_correction) == mr12_history
assert codec.decode_json(codec.encode_json(mr12_history), EligibilityAssertions) == mr12_history
mr12_tick = timedelta(microseconds=1)
mr12_rows = []
for label, boundary in (
    ("Original knowledge", mr12_original.ts_knowledge),
    ("Correction knowledge", mr12_correction.ts_knowledge),
    ("Corrected valid start", mr12_correction.valid_from),
    ("Corrected valid end", mr12_correction.valid_to),
):
    for offset in (-1, 0, 1):
        valid_at = mr12_correction.valid_from
        known_at = mr12_correction.ts_knowledge
        if "knowledge" in label:
            known_at = boundary + offset * mr12_tick
        else:
            valid_at = boundary + offset * mr12_tick
        selected = mr12_history.select(
            mr12_original.instrument_id, valid_at=valid_at, known_at=known_at
        )
        if "knowledge" in label:
            expected = (
                "unknown"
                if known_at < mr12_original.ts_knowledge
                else mr12_original
                if known_at < mr12_correction.ts_knowledge
                else mr12_correction
            )
        else:
            expected = (
                mr12_correction
                if mr12_correction.valid_from <= valid_at < mr12_correction.valid_to
                else "unknown"
            )
        assert selected == expected
        mr12_rows.append(
            (
                label,
                offset,
                valid_at,
                known_at,
                selected.state if isinstance(selected, EligibilityAssertion) else selected,
                selected.evidence.identity
                if isinstance(selected, EligibilityAssertion)
                else "No assertion",
            )
        )
NotebookDisplay.show_table(
    ("Boundary", "Offset (µs)", "Valid at", "Known at", "Membership", "Evidence"), mr12_rows
)

mr12_universe_rows = []
for cutoff in (mr12_correction.ts_knowledge - mr12_tick, mr12_correction.ts_knowledge):
    selected_members = []
    for member in mr12_history.catalog.instruments:
        selected = mr12_history.select(
            member.instrument_id, valid_at=mr12_correction.valid_from, known_at=cutoff
        )
        if cutoff < mr12_correction.ts_knowledge:
            assert selected == mr12_before.select(
                member.instrument_id, valid_at=mr12_correction.valid_from, known_at=cutoff
            )
        if isinstance(selected, EligibilityAssertion) and selected.state == "included":
            selected_members.append(member.symbol)
    mr12_universe_rows.append((cutoff, selected_members))
NotebookDisplay.show_table(("Knowledge cutoff", "Included universe at 02:00"), mr12_universe_rows)

Boundary,Offset (µs),Valid at,Known at,Membership,Evidence
Original knowledge,-1,2024-06-01 02:00:00+00:00,2024-05-31 23:59:59.999999+00:00,unknown,No assertion
Original knowledge,0,2024-06-01 02:00:00+00:00,2024-06-01 00:00:00+00:00,included,synthetic:mr11:sample-announcement-v1
Original knowledge,1,2024-06-01 02:00:00+00:00,2024-06-01 00:00:00.000001+00:00,included,synthetic:mr11:sample-announcement-v1
Correction knowledge,-1,2024-06-01 02:00:00+00:00,2024-06-01 11:59:59.999999+00:00,included,synthetic:mr11:sample-announcement-v1
Correction knowledge,0,2024-06-01 02:00:00+00:00,2024-06-01 12:00:00+00:00,excluded,synthetic:mr12:listing-correction-v2
Correction knowledge,1,2024-06-01 02:00:00+00:00,2024-06-01 12:00:00.000001+00:00,excluded,synthetic:mr12:listing-correction-v2
Corrected valid start,-1,2024-06-01 01:59:59.999999+00:00,2024-06-01 12:00:00+00:00,unknown,No assertion
Corrected valid start,0,2024-06-01 02:00:00+00:00,2024-06-01 12:00:00+00:00,excluded,synthetic:mr12:listing-correction-v2
Corrected valid start,1,2024-06-01 02:00:00.000001+00:00,2024-06-01 12:00:00+00:00,excluded,synthetic:mr12:listing-correction-v2
Corrected valid end,-1,2024-06-01 23:59:59.999999+00:00,2024-06-01 12:00:00+00:00,excluded,synthetic:mr12:listing-correction-v2


Knowledge cutoff,Included universe at 02:00
2024-06-01 11:59:59.999999+00:00,"['BTCUSDT', 'ETHUSDT']"
2024-06-01 12:00:00+00:00,['ETHUSDT']


**Observed output:** BTC is unknown before the original knowledge time, included
until the correction becomes known, then excluded on the corrected valid interval.
At 02:00 the earlier universe remains BTC+ETH even after the correction is appended;
at the noon knowledge cutoff it becomes ETH only. At the corrected valid endpoint
BTC is unknown. ETH's existing exclusion remains retained beyond catalog validity.
These are synthetic membership decisions, not validated historical trading permissions.

**Required failures:** remove correction linkage, name a missing predecessor, append
a competing branch, and query with a naive decision clock. Also show unknown versus
unsupported explicitly, then confirm a valid selection still works.

In [19]:
mr12_failures = []
for label, retained, invalid in (
    ("Unlinked conflicting exclusion", mr12_before, replace(mr12_correction, supersedes=None)),
    ("Missing correction predecessor", mr12_before, replace(mr12_correction, supersedes="0" * 64)),
    (
        "Competing correction branch",
        mr12_history,
        replace(mr12_correction, reason="Competing branch"),
    ),
):
    try:
        retained.append(invalid)
    except ValueError as error:
        mr12_failures.append((label, str(error)))
    else:
        raise AssertionError(f"invalid correction admitted: {label}")
try:
    mr12_history.select(
        mr12_original.instrument_id,
        valid_at=mr12_correction.valid_from,
        known_at=datetime(2024, 6, 1, 12),
    )
except TypeError as error:
    mr12_failures.append(("Naive decision clock", str(error)))
else:
    raise AssertionError("naive decision clock admitted")
assert len(mr12_failures) == 4
NotebookDisplay.show_table(("Required failure", "Observed rejection"), mr12_failures)
mr12_missing = InstrumentId.from_canonical_text("33333333-3333-4333-8333-333333333333")
mr12_unknown = mr12_history.select(
    mr12_original.instrument_id,
    valid_at=mr12_correction.valid_to,
    known_at=mr12_correction.ts_knowledge,
)
mr12_unsupported = mr12_history.select(
    mr12_missing, valid_at=mr12_original.valid_from, known_at=mr12_correction.ts_knowledge
)
assert mr12_unknown == "unknown" and mr12_unsupported == "unsupported"
assert (
    mr12_history.select(
        mr12_original.instrument_id,
        valid_at=mr12_correction.valid_from,
        known_at=mr12_correction.ts_knowledge,
    )
    == mr12_correction
)
NotebookDisplay.show_table(
    ("Unavailable membership", "Outcome"),
    [
        ("Catalog member without a covering known assertion", mr12_unknown),
        ("Member absent from the selected catalog", mr12_unsupported),
    ],
)

Required failure,Observed rejection
Unlinked conflicting exclusion,contradictory eligibility overlap: 11111111-1111-4111-8111-111111111111
Missing correction predecessor,"correction requires a retained same-member predecessor, later knowledge and distinct evidence"
Competing correction branch,eligibility correction history cannot branch
Naive decision clock,known_at must be a timezone-aware datetime


Unavailable membership,Outcome
Catalog member without a covering known assertion,unknown
Member absent from the selected catalog,unsupported


**Interpretation and limits:** the selector retains reasons and evidence in its
included/excluded results and never upgrades unknown or unsupported to inclusion.
The old immutable history is unchanged; version-2 serialization preserves the
predecessor and its link. Version-1 files remain readable, but corrections require
version 2. Corrections form one linear chain per assertion; branching, dangling
links and conflicting active assertions fail closed. This small in-memory selection
has no dataset publication, source authentication or large-history performance claim.

**Next research question:** can normalized observations, revisions, membership and
coverage jointly pass a declared-use quality check? That belongs to **MR 13**.
The [MR 12 evidence record](../docs/roadmap/mr-12-implementation.md) identifies this run.

## 13. Can these trades pass a declared-use quality gate? (NB 13)

**Question:** which defects invalidate a partition, and which missing evidence can
an explicitly exploratory study retain? **Sample:** reuse the tiny synthetic
Binance originals/duplicates and the membership history from the preceding sections.
A five-minute observed-silence tolerance is an exercise policy, not P1 acceptance.
No download, whole-sample clock repair or historical completeness assertion is made.

**Action:** compare exploratory and causal use, inspect gap/invalid/duplicate counts,
then request canonical bytes through the publication gate. The bytes retain full
revision and eligibility histories, source clocks and explicit degraded scope.


In [20]:
# project library imports
from alpha_research.data.quality import QualityPolicy, QualityReport
from alpha_research.pipelines.dataset import validate_saved_partition

mr13_policy = QualityPolicy(
    use="exploratory", known_at=original_request.manifest.job.start, max_gap=timedelta(minutes=5)
)
mr13_inputs = archive_original.occurrences
mr13_report = mr13_policy.validate(original_request, mr13_inputs, eligibility=mr12_history)
assert (mr13_report.invalid_count, mr13_report.gap_count, mr13_report.duplicate_count) == (0, 1, 1)
assert mr13_report.scope == "exploratory_degraded"
mr13_canonical = mr13_report.require_canonical()
assert (
    mr13_canonical
    == mr13_policy.validate(
        original_request, tuple(reversed(mr13_inputs)), eligibility=mr12_history
    ).require_canonical()
)
assert codec.decode_json(codec.encode_json(mr13_report), QualityReport) == mr13_report
mr13_strict = replace(mr13_policy, use="causal").validate(
    original_request, mr13_inputs, eligibility=mr12_history
)
assert mr13_strict.canonical is None
NotebookDisplay.show_table(
    ("Declared use", "Quality scope", "Invalid inputs", "Observed gaps", "Duplicates"),
    [
        (label, report.scope, report.invalid_count, report.gap_count, report.duplicate_count)
        for label, report in (("Exploratory", mr13_report), ("Causal", mr13_strict))
    ],
)
NotebookDisplay.show_table(("Retained limitation",), [(reason,) for reason in mr13_report.reasons])
print("Canonical bytes / SHA-256:", len(mr13_canonical), hashlib.sha256(mr13_canonical).hexdigest())

Declared use,Quality scope,Invalid inputs,Observed gaps,Duplicates
Exploratory,exploratory_degraded,0,1,1
Causal,rejected,0,1,1


Retained limitation
catalog units are authored research assumptions
observed gaps: 1; disconnects: 0
original trade availability unknown
recorder coverage unknown


Canonical bytes / SHA-256: 11939 b6edc61654aa15f38fe95b3846b1dafb7a05ed1559f0457b98ee13a3a16f1290


**Observed output:** the duplicate contributes once with both raw origins. The
remaining interval exceeds five minutes without an observation. These archives
have unknown recorder coverage and original availability even if a different gap
tolerance would report zero gaps. The authored catalog is not historical evidence.
Exploratory bytes retain these limitations; causal publication is refused.

**Required failures:** add the independent ETH occurrence, omit a correction link,
rewrite a captured knowledge clock, and supply NaN/zero prices and zero trade
quantity through saved archives. Reader failures retain their row and explicitly
mark the unread remainder unvalidated. Original raw objects remain stored.


In [21]:
mr13_failures = []
for label, report in (
    ("Required coverage / gap", mr13_strict),
    (
        "Wrong instrument",
        mr13_policy.validate(
            original_request, (*mr13_inputs, mr09_items[2]), eligibility=mr12_history
        ),
    ),
    (
        "Conflicting correction",
        mr13_policy.validate(original_request, (*mr13_inputs, revised), eligibility=mr12_history),
    ),
):
    try:
        report.require_canonical()
    except ValueError:
        mr13_failures.append(
            (label, report.invalid_count, report.gap_count, "; ".join(report.reasons))
        )
    else:
        raise AssertionError(f"publication gate accepted {label}")

mr13_capture = captured_versions[0]
mr13_shifted = replace(
    mr13_capture,
    event=replace(
        mr13_capture.event,
        provenance=replace(
            mr13_capture.event.provenance,
            ts_knowledge=mr13_capture.event.provenance.ts_knowledge + timedelta(seconds=10),
        ),
    ),
)
mr13_captured_history = replace(mr12_history, catalog=captured_binding.manifest.catalog)
mr13_clock_report = mr13_policy.validate(
    captured_binding, (mr13_shifted,), eligibility=mr13_captured_history
)
assert mr13_clock_report.canonical is None and mr13_clock_report.invalid_count == 1
assert "original bytes/clocks" in "; ".join(mr13_clock_report.reasons)
assert mr13_capture.event.provenance.ts_knowledge == original_version.ts_knowledge
mr13_failures.append(
    (
        "Future whole-sample clock shift",
        1,
        mr13_clock_report.gap_count,
        "; ".join(mr13_clock_report.reasons),
    )
)

with TemporaryDirectory(prefix="nb13-quality-") as temporary:
    local = Path(temporary)
    store = RawStore(local)
    for label, content in (
        ("NaN price", mr09_first.replace(b"100.00", b"NaN")),
        ("Zero price", mr09_first.replace(b"100.00", b"0")),
        ("Zero trade quantity", mr09_first.replace(b",2.0,", b",0,")),
    ):
        path = local / "source.zip"
        member = ZipInfo("BTCUSDT-trades-2024-06-01.csv")
        member.compress_type = ZIP_DEFLATED
        with ZipFile(path, "w") as archive:
            archive.writestr(member, content)
        origin = RawObject.from_file(
            path,
            request=original_request,
            retrieval_reference="synthetic:nb13",
            ts_received=datetime(2026, 9, 16, tzinfo=UTC),
            ts_ingested=datetime(2026, 9, 16, tzinfo=UTC),
        )
        saved_path = store.publish(path, descriptor=origin, request=original_request)
        state = AcquisitionManifest(run=original_run, completed=(origin,))
        report = validate_saved_partition(
            state, store, max_bytes=4096, policy=mr13_policy, eligibility=mr12_history
        )
        assert report.invalid_count == 1 and report.canonical is None
        assert saved_path.read_bytes() == path.read_bytes()
        try:
            report.require_canonical()
        except ValueError:
            mr13_failures.append(
                (label, report.invalid_count, report.gap_count, "; ".join(report.reasons))
            )
        else:
            raise AssertionError(f"publication gate accepted {label}")
assert len(mr13_failures) == 7
NotebookDisplay.show_table(
    ("Failure", "Invalid inputs", "Observed gaps", "Retained reasons"), mr13_failures
)

Failure,Invalid inputs,Observed gaps,Retained reasons
Required coverage / gap,0,1,catalog units are authored research assumptions; observed gaps: 1; disconnects: 0; original trade availability unknown; recorder coverage unknown
Wrong instrument,1,1,catalog units are authored research assumptions; input 2: occurrence differs from selected partition/instrument/units; observed gaps: 1; disconnects: 0; original trade availability unknown; recorder coverage unknown
Conflicting correction,0,1,"catalog units are authored research assumptions; observed gaps: 1; disconnects: 0; original trade availability unknown; recorder coverage unknown; revision admission: unresolved trade revisions: fcf81aaf5c297bdcd4b6acc78dcb9704d3e47e4f0e962ea5bfd0a5435f6da0d9, 06d4f503e5f06bc6965ef5d7928daacce088114d314a1743616b6cb5e864e4ef"
Future whole-sample clock shift,1,0,catalog units are authored research assumptions; input 0: canonical capture differs from original bytes/clocks; no admissible trades in selected interval; recorder coverage unknown
NaN price,1,1,catalog units are authored research assumptions; input incomplete; unread remainder not validated: invalid Binance trade at row 0: price must be finite; no admissible trades in selected interval; observed gaps: 1; disconnects: 0; recorder coverage unknown
Zero price,1,1,catalog units are authored research assumptions; input incomplete; unread remainder not validated: invalid Binance trade at row 0: price must be positive; no admissible trades in selected interval; observed gaps: 1; disconnects: 0; recorder coverage unknown
Zero trade quantity,1,1,catalog units are authored research assumptions; input 0: trade quantity must be positive; no admissible trades in selected interval; observed gaps: 1; disconnects: 0; recorder coverage unknown


**Interpretation and limits:** all seven refusal cases retain their reasons.
Prices must be finite and positive on the authored tick grid. Trade base quantity
must be positive; the existing depth-update codec still allows zero to mean deletion.
Publisher quote amounts remain exact source fields; no undocumented rounding rule
is used to replace them with price × quantity. Source order is checked per raw
object and economic revisions retain every predecessor and effective availability.
Excluded and unknown membership intervals remain annotated, including intervals
without trades. Causal use cannot admit their unresolved requirements.

This bounded in-memory gate supplies canonical bytes only for explicitly degraded
exploration. It cannot certify complete historical coverage, authenticate membership,
publish a dataset snapshot or establish a P1 result. A terminal reader failure gives
an observed invalid count, not a count of its unread remainder. See the
[MR 13 implementation record](../docs/roadmap/mr-13-implementation.md).

**Next research question:** section 14 identifies the admitted partitions, policies
and resolved references together and verifies their saved bytes.


## 14. Do the same saved inputs identify the same dataset? (NB 14)

**Question:** can I distinguish a reproducible rebuild from a changed policy or
reference, without making identity depend on this notebook's directory or run time?
**Sample:** a tiny deterministic synthetic Binance ZIP containing a duplicate trade,
the resolved request from section 9 and complete membership history from section 12.
The one-hour and five-minute gap tolerances are teaching policies, not P1 choices.

**Action:** assemble twice, change only the gap policy, save working canonical files
and a manifest, then read each with its explicit expected identity. The reader
verifies raw bytes and reproduces quality admission. These are working files;
section 15 adds atomic snapshot publication.


In [22]:
# project library imports
from alpha_research.data.datasets import DatasetManifest
from alpha_research.pipelines.dataset import assemble_dataset

mr14_policy = replace(mr13_policy, max_gap=timedelta(hours=1))
mr14_builds = []
mr14_hash_rows = []
with TemporaryDirectory(prefix="nb14-build-") as temporary:
    local = Path(temporary)
    store = RawStore(local)
    path = local / "source.zip"
    member = ZipInfo("BTCUSDT-trades-2024-06-01.csv")
    member.compress_type = ZIP_DEFLATED
    with ZipFile(path, "w") as archive:
        archive.writestr(member, mr09_first * 2)
    origin = RawObject.from_file(
        path,
        request=original_request,
        retrieval_reference="synthetic:nb14",
        ts_received=datetime(2026, 9, 16, tzinfo=UTC),
        ts_ingested=datetime(2026, 9, 16, tzinfo=UTC),
    )
    store.publish(path, descriptor=origin, request=original_request)
    mr14_raw_bytes = path.read_bytes()
    mr14_acquisition = AcquisitionManifest(run=original_run, completed=(origin,))
    for label, policy in (
        ("First build", mr14_policy),
        ("Same saved inputs", mr14_policy),
        ("Gap policy changed", replace(mr14_policy, max_gap=timedelta(minutes=5))),
    ):
        manifest, payloads = assemble_dataset(
            (mr14_acquisition,),
            store,
            max_bytes=262_144,
            policy=policy,
            eligibility=mr12_history,
            corrections={},
        )
        for partition, payload in zip(manifest.partitions, payloads, strict=True):
            (local / partition.sha256).write_bytes(payload)
        (local / "dataset.json").write_bytes(codec.encode_json(manifest))
        assert (
            DatasetManifest.read(
                local / "dataset.json",
                identity=manifest.identity,
                partitions=local,
                store=store,
                max_bytes=262_144,
            )
            == manifest
        )
        mr14_builds.append((manifest, payloads))
        mr14_hash_rows.append(
            (label, manifest.identity, manifest.partitions[0].sha256, manifest.scope)
        )
assert mr14_builds[0] == mr14_builds[1]
assert mr14_builds[0][0].identity != mr14_builds[2][0].identity
NotebookDisplay.show_table(
    ("Build", "Dataset SHA-256", "Canonical SHA-256", "Scope"), mr14_hash_rows
)
print("Resolved job/universe:", mr14_acquisition.run.identity)
print("Catalog:", mr14_acquisition.run.catalog.identity)
print("Eligibility history:", hashlib.sha256(codec.encode_json(mr12_history)).hexdigest())
print("Normalization implementation:", mr14_builds[0][0].normalization_identity)

Build,Dataset SHA-256,Canonical SHA-256,Scope
First build,496c64f5376dc9b495cf7e8def9d1761a75c6cb7486a0b9578f9028a7a1349df,bf29ab5ea3805b47c76310a865d49636c14a81fb119bbbd202851a96f5797b33,exploratory_degraded
Same saved inputs,496c64f5376dc9b495cf7e8def9d1761a75c6cb7486a0b9578f9028a7a1349df,bf29ab5ea3805b47c76310a865d49636c14a81fb119bbbd202851a96f5797b33,exploratory_degraded
Gap policy changed,adc2882b24f9bec695f60c635561518fae43685866cd3797d0e216be14ccab9f,cd32e8c1d23add175279f825418d24b634d44dd2cf2c4e7038c5e8c083bed0bf,exploratory_degraded


Resolved job/universe: 4528da28f8f0b6c5320968aac54daefc29941da51548671548288d8097733810
Catalog: 8fbddf0bbffea1168f4bd907025219863789f99f12c624ab54350c95c8fbc69b
Eligibility history: e57eea3a10b7a16dfec3d5406684e404cc5d2ca62b399afb8b8f1d4bfbfd5486
Normalization implementation: 1092bc33c6947bf3e50a978d5d6bcea26c683f3f0b69089cdc601fd554e0e50a


**Observed output:** the first two dataset and canonical hashes match. Changing the
gap policy changes both, while the scope stays explicitly exploratory degraded.
Identity also binds ordered partitions, embedded raw manifests, actual selected
instruments/catalog, full eligibility, normalization implementation/schema and
correction/availability policy. A mutable job name alone is insufficient.

**Required failures:** in a different directory, try an unknown manifest version,
an unsupported causal scope, changed identity, missing canonical bytes and missing
raw bytes. Restore the same bytes and verify recovery. Build wall-clock times and
local paths have no manifest fields; original acquisition/evidence clocks remain
semantic provenance and must not be replaced by rebuild time.


In [23]:
mr14_failures = []
mr14_manifest, mr14_payloads = mr14_builds[0]
with TemporaryDirectory(prefix="nb14-relocated-") as temporary:
    local = Path(temporary)
    store = RawStore(local)
    path = local / "source.zip"
    path.write_bytes(mr14_raw_bytes)
    raw_path = store.publish(path, descriptor=origin, request=original_request)
    canonical_path = local / mr14_manifest.partitions[0].sha256
    canonical_path.write_bytes(mr14_payloads[0])
    manifest_path = local / "dataset.json"
    for label, field, value in (
        ("Unknown manifest version", "schema_version", "99"),
        ("Unsupported causal scope", "scope", "causal"),
        ("Changed normalization identity", "normalization_identity", "a" * 64),
    ):
        changed = json.loads(codec.encode_json(mr14_manifest))
        changed[field] = value
        manifest_path.write_bytes(codec.encode_json(changed))
        try:
            DatasetManifest.read(
                manifest_path,
                identity=mr14_manifest.identity,
                partitions=local,
                store=store,
                max_bytes=262_144,
            )
        except ValueError as error:
            mr14_failures.append((label, str(error)))
        else:
            raise AssertionError(f"manifest reader accepted {label}")
    manifest_path.write_bytes(codec.encode_json(mr14_manifest))
    for label, missing_path in (
        ("Missing canonical bytes", canonical_path),
        ("Missing raw bytes", raw_path),
    ):
        original_bytes = missing_path.read_bytes()
        missing_path.chmod(0o644)
        missing_path.unlink()
        try:
            DatasetManifest.read(
                manifest_path,
                identity=mr14_manifest.identity,
                partitions=local,
                store=store,
                max_bytes=262_144,
            )
        except (ValueError, FileNotFoundError) as error:
            mr14_failures.append((label, str(error)))
        else:
            raise AssertionError(f"manifest reader accepted {label}")
        missing_path.write_bytes(original_bytes)
    assert (
        DatasetManifest.read(
            manifest_path,
            identity=mr14_manifest.identity,
            partitions=local,
            store=store,
            max_bytes=262_144,
        )
        == mr14_manifest
    )
assert len(mr14_failures) == 5
NotebookDisplay.show_table(("Refused input", "Diagnostic"), mr14_failures)
print("Relocated and restored dataset:", mr14_manifest.identity)

Refused input,Diagnostic
Unknown manifest version,Invalid enum value '99' - at `$.schema_version`
Unsupported causal scope,Invalid enum value 'causal' - at `$.scope`
Changed normalization identity,dataset manifest identity mismatch
Missing canonical bytes,[Errno 2] No such file or directory: '/var/folders/30/8y6v8_j93_d8k3xzh930ghqw0000gn/T/nb14-relocated-drv94gcz/bf29ab5ea3805b47c76310a865d49636c14a81fb119bbbd202851a96f5797b33'
Missing raw bytes,partition publication refused: catalog units are authored research assumptions; input incomplete; unread remainder not validated: [Errno 2] No such file or directory: '/var/folders/30/8y6v8_j93_d8k3xzh930ghqw0000gn/T/nb14-relocated-drv94gcz/5dc93c12483e485195b8ef2c4d2f1fdf8933477ad91d171dc782eb9a7ad7c299'; no admissible trades in selected interval; recorder coverage unknown


Relocated and restored dataset: 496c64f5376dc9b495cf7e8def9d1761a75c6cb7486a0b9578f9028a7a1349df


**Interpretation and limits:** all five invalid/missing-input cases fail visibly;
restoring the same bytes in another directory recovers the same identity. The reader
also replays admission from verified raw archives, so a newly hashed file cannot
claim better quality or silently discard history. A digest proves content binding,
not historical source authenticity, complete coverage or empirical membership.
This is a bounded in-memory reference reader over the supported Binance trade path;
it does not provide dataset queries, atomic publication, M1 completion or a P1 result.
See the [MR 14 implementation record](../docs/roadmap/mr-14-implementation.md).

**Next research question:** section 15 publishes a completed snapshot atomically
and checks that it remains reopenable after an interrupted later build.


## 15. What can I reopen after an interrupted publication? (NB 15)

**Question:** can a reader see a half-written dataset, and does a later failed build
make an earlier snapshot disappear? **Sample:** reuse section 14's synthetic raw
archive, complete manifest, canonical bytes and explicit degraded quality policies.
The temporary snapshot directory and retained raw store are separate.

**Action:** inject an I/O failure at the atomic directory rename, attempt an explicit-ID
read, then publish through the dataset pipeline and reopen. Repeat the failure with
a changed-policy dataset while checking the earlier snapshot. Finally publish the
new dataset and reopen both. The fault is local and deterministic; the retained
regressions also terminate actual child processes without Python cleanup.


In [24]:
# standard library imports
from unittest.mock import patch

# project library imports
from alpha_research.data.snapshots import SnapshotStore
from alpha_research.pipelines.dataset import publish_dataset

mr15_observations = []
with TemporaryDirectory(prefix="nb15-publication-") as temporary:
    local = Path(temporary)
    raw_directory = local / "raw"
    raw_directory.mkdir()
    destination = local / "snapshots"
    destination.mkdir()
    store = RawStore(raw_directory)
    snapshots = SnapshotStore(destination, store)
    source = local / "source.zip"
    source.write_bytes(mr14_raw_bytes)
    store.publish(source, descriptor=mr14_acquisition.completed[0], request=original_request)
    mr15_old_identity = mr14_builds[0][0].identity
    for label, (manifest, payloads) in (
        ("First dataset", mr14_builds[0]),
        ("Changed policy", mr14_builds[2]),
    ):
        with patch(
            "alpha_research.data.snapshots.os.rename",
            side_effect=OSError("synthetic interruption before atomic rename"),
        ):
            try:
                snapshots.publish(manifest, payloads, max_bytes=262_144)
            except OSError as error:
                print(label, "publication refused:", error)
            else:
                raise AssertionError("injected interruption did not reach publication")
        try:
            snapshots.read(manifest.identity, max_bytes=262_144)
        except FileNotFoundError:
            mr15_observations.append(
                (label, "After interruption", "Unavailable", manifest.identity)
            )
        else:
            raise AssertionError("reader exposed a partial snapshot")
        assert not list(destination.glob(".staging-*"))
        if label == "Changed policy":
            assert snapshots.read(mr15_old_identity, max_bytes=262_144) == mr14_builds[0][0]
            mr15_observations.append(
                ("First dataset", "While new build failed", "Readable", mr15_old_identity)
            )
        published = publish_dataset(
            (mr14_acquisition,),
            store,
            destination,
            max_bytes=262_144,
            policy=manifest.partitions[0].policy,
            eligibility=mr12_history,
            corrections={},
        )
        assert published == manifest.identity
        reopened = snapshots.read(published, max_bytes=262_144)
        assert reopened == manifest and reopened.scope == "exploratory_degraded"
        assert snapshots.publish(manifest, payloads, max_bytes=262_144) == published
        mr15_observations.append((label, "Published and retried", "Readable", published))
    assert snapshots.read(mr15_old_identity, max_bytes=262_144) == mr14_builds[0][0]
    mr15_observations.append(
        ("First dataset", "After new publication", "Readable", mr15_old_identity)
    )
    try:
        snapshots.read("latest", max_bytes=262_144)
    except ValueError as error:
        print("Mutable tag refused:", error)
    else:
        raise AssertionError("reader silently resolved a mutable latest tag")
    assert {path.name for path in destination.iterdir()} == {
        mr14_builds[0][0].identity,
        mr14_builds[2][0].identity,
    }
NotebookDisplay.show_table(
    ("Dataset", "Reader checkpoint", "Result", "Explicit snapshot SHA-256"), mr15_observations
)

First dataset publication refused: synthetic interruption before atomic rename


Changed policy publication refused: synthetic interruption before atomic rename
Mutable tag refused: snapshot requires an explicit lowercase SHA-256 identity


Dataset,Reader checkpoint,Result,Explicit snapshot SHA-256
First dataset,After interruption,Unavailable,496c64f5376dc9b495cf7e8def9d1761a75c6cb7486a0b9578f9028a7a1349df
First dataset,Published and retried,Readable,496c64f5376dc9b495cf7e8def9d1761a75c6cb7486a0b9578f9028a7a1349df
Changed policy,After interruption,Unavailable,adc2882b24f9bec695f60c635561518fae43685866cd3797d0e216be14ccab9f
First dataset,While new build failed,Readable,496c64f5376dc9b495cf7e8def9d1761a75c6cb7486a0b9578f9028a7a1349df
Changed policy,Published and retried,Readable,adc2882b24f9bec695f60c635561518fae43685866cd3797d0e216be14ccab9f
First dataset,After new publication,Readable,496c64f5376dc9b495cf7e8def9d1761a75c6cb7486a0b9578f9028a7a1349df


**Observed output:** neither interrupted build has a readable snapshot ID. After
publication, both IDs reopen with the same admitted bytes and references. Repeating
publication preserves the identity. The earlier dataset stays readable during the
later failure and after the changed-policy snapshot is published. `latest` fails
explicitly; a run must retain its selected SHA-256.

**Interpretation and limits:** snapshot publication is one local directory rename
after file synchronization and complete manifest verification. A failure after the
rename can leave a complete visible snapshot with durability still unacknowledged;
retrying the same ID verifies it and synchronizes the parent. An abruptly killed
writer can leave hidden staging directories; readers cannot select them. No automatic
cleanup, migration, query service or latest-tag resolver is introduced.

The original raw store must remain available and protected from mutation, just as
the published manifest and canonical files must. Publication does not upgrade
exploratory degraded quality or supply missing historical evidence. See the
[MR 15 implementation record](../docs/roadmap/mr-15-implementation.md).

**Next research question — MR 16:** how does the installed dataset command report
normalization, quality, persistence and reopening from saved acquisitions? Section 16 exercises that command; empirical source-evidence limits remain open.


## 16. Can I publish and inspect saved trades from an installed command? (NB 16)

**Question:** how do I distinguish accepted exploratory observations, rejected inputs,
and an interrupted save without inspecting implementation code? **Sample:** reuse the
small synthetic Binance acquisition and explicit policy/eligibility from section 14.
No network request or empirical source claim is made.

**Action:** run `alpha-dataset publish` in a temporary directory outside the checkout,
repeat the same range twice, then run `alpha-dataset inspect` with the returned ID.
Observe flushed stages and compact quality counts; canonical observations are not
printed. The complete command contract is in the [component guide](../src/alpha_research/data/README.md#publish-and-inspect-from-the-command-line-mr-16).


In [25]:
# standard library imports
from shutil import which

mr16_executable = which("alpha-dataset")
assert mr16_executable is not None, "Refresh the pinned project installation first."
print("Installed command:", mr16_executable)


class DatasetCommandExercise:
    """Invoke the installed dataset command and retain compact, inspected JSON reports."""

    @staticmethod
    def run(
        arguments: list[str], expected: int = 0, child: Path | None = None
    ) -> list[dict[str, object]]:
        """Execute outside the checkout and check success or the requested controlled failure.

        Args:
            arguments: Explicit command inputs including retained file paths.
            expected: Required process exit code.
            child: Optional retained abrupt-interruption fixture.

        Returns:
            Parsed progress and terminal records, excluding raw observations.
        """
        command = [mr16_executable]
        if child is not None:
            command = [str(Path(mr16_executable).with_name("python")), "-I", str(child)]
        result = subprocess.run(
            [*command, *arguments],
            cwd=mr16_local,
            capture_output=True,
            text=True,
            check=False,
            timeout=30,
        )
        assert result.returncode == expected, (result.stdout, result.stderr)
        records = [json.loads(line) for line in (result.stdout + result.stderr).splitlines()]
        if expected == 0:
            assert records[-1]["status"] == "complete"
        else:
            assert not any(record["status"] == "complete" for record in records)
        return records


mr16_temporary = TemporaryDirectory(prefix="nb16-command-")
mr16_local = Path(mr16_temporary.name)
mr16_raw = mr16_local / "raw"
mr16_raw.mkdir()
mr16_snapshots = mr16_local / "snapshots"
mr16_snapshots.mkdir()
mr16_source = mr16_local / "source.zip"
mr16_source.write_bytes(mr14_raw_bytes)
RawStore(mr16_raw).publish(
    mr16_source, descriptor=mr14_acquisition.completed[0], request=original_request
)
mr16_manifest_path = mr16_local / "acquisition.json"
mr14_acquisition.save(mr16_manifest_path)
mr16_policy_path = mr16_local / "policy.json"
mr16_policy = mr14_builds[0][0].partitions[0].policy
mr16_policy_path.write_bytes(codec.encode_json(mr16_policy))
(mr16_local / "eligibility.json").write_bytes(codec.encode_json(mr12_history))
mr16_common = [
    "--store",
    str(mr16_raw),
    "--snapshots",
    str(mr16_snapshots),
    "--max-bytes",
    "262144",
]
mr16_publish = [
    "publish",
    *mr16_common,
    "--policy",
    str(mr16_policy_path),
    "--eligibility",
    str(mr16_local / "eligibility.json"),
    "--manifests",
    str(mr16_manifest_path),
]
mr16_progress = DatasetCommandExercise.run(mr16_publish)
mr16_identity = str(mr16_progress[-1]["snapshot_id"])
mr16_repeat = DatasetCommandExercise.run([*mr16_publish, str(mr16_manifest_path)])
mr16_inspect = ["inspect", *mr16_common, "--snapshot-id", mr16_identity]
mr16_reopened = DatasetCommandExercise.run(mr16_inspect)
assert mr16_repeat[-1]["snapshot_id"] == mr16_reopened[-1]["snapshot_id"] == mr16_identity
NotebookDisplay.show_table(
    ("Stage", "Status", "Quality scope", "Invalid / gaps / duplicates"),
    [
        (
            record["stage"],
            record["status"],
            record.get("scope", ""),
            " / ".join(
                str(record.get(field, ""))
                for field in ("invalid_count", "gap_count", "duplicate_count")
            ),
        )
        for record in mr16_progress
    ],
)
print("Published, repeated and reopened:", mr16_identity)
print(
    "Coverage limitations:",
    next(record["reasons"] for record in mr16_progress if "reasons" in record),
)

Installed command: /Users/lxg/Documents/GitHub/Alpha Research/alpha-research-lab-private/.venv/bin/alpha-dataset


Stage,Status,Quality scope,Invalid / gaps / duplicates
configuration,started,,/ /
normalize,started,,/ /
normalize,validated,exploratory_degraded,0 / 0 / 1
publish,started,,/ /
publish,persisted,,/ /
reopen,started,,/ /
reopen,complete,exploratory_degraded,/ /


Published, repeated and reopened: 496c64f5376dc9b495cf7e8def9d1761a75c6cb7486a0b9578f9028a7a1349df
Coverage limitations: ['catalog units are authored research assumptions', 'original trade availability unknown', 'recorder coverage unknown']


**Observed output:** publication reaches persistence and verified reopening.
Repeating an identical acquisition selection retains the same snapshot identity.
The report still says **exploratory degraded**, including unknown recorder coverage
and original availability. Duplicate counts describe observed economic duplicates,
not proof of source completeness.

**Required failures:** supply wrong CSV schema, another instrument's archive, a trade
outside its UTC day, an off-grid price, and a causal policy requiring missing coverage.
Then terminate an actual command process before rename. Check that only the earlier
completed snapshot is readable, retry, and reopen both IDs.


In [26]:
mr16_failures = []
try:
    for label, row, member_symbol in (
        ("Wrong CSV schema", b"wrong,columns\n", "BTCUSDT"),
        ("Wrong archive instrument", b"12,100.00,2.0,200,1717200000000,true,true\n", "ETHUSDT"),
        ("Wrong UTC day", b"12,100.00,2.0,200,1717286400000,true,true\n", "BTCUSDT"),
        ("Off-grid price units", b"12,100.001,2.0,200,1717200000000,true,true\n", "BTCUSDT"),
    ):
        with ZipFile(mr16_source, "w", compression=ZIP_DEFLATED) as archive:
            archive.writestr(f"{member_symbol}-trades-2024-06-01.csv", row)
        descriptor = RawObject.from_file(
            mr16_source,
            request=original_request,
            retrieval_reference="synthetic:nb16:failure",
            ts_received=mr14_acquisition.completed[0].ts_received,
            ts_ingested=mr14_acquisition.completed[0].ts_ingested,
        )
        RawStore(mr16_raw).publish(mr16_source, descriptor=descriptor, request=original_request)
        replace(mr14_acquisition, completed=(descriptor,)).save(mr16_manifest_path)
        records = DatasetCommandExercise.run(mr16_publish, expected=3)
        quality = next(record for record in records if record["status"] == "rejected")
        assert quality["invalid_count"] == 1
        mr16_failures.append(
            (label, 3, quality["scope"], quality["invalid_count"], quality["gap_count"])
        )
    mr14_acquisition.save(mr16_manifest_path)
    mr16_policy_path.write_bytes(
        codec.encode_json(replace(mr16_policy, use="causal", max_gap=timedelta(seconds=1)))
    )
    records = DatasetCommandExercise.run(mr16_publish, expected=3)
    quality = next(record for record in records if record["status"] == "rejected")
    assert quality["gap_count"] > 0
    mr16_failures.append(
        (
            "Required coverage with a gap",
            3,
            quality["scope"],
            quality["invalid_count"],
            quality["gap_count"],
        )
    )
    mr16_policy_path.write_bytes(
        codec.encode_json(replace(mr16_policy, max_gap=timedelta(seconds=2)))
    )
    stopped = DatasetCommandExercise.run(
        mr16_publish, expected=75, child=root / "tests/fixtures/mr16/interrupt_command.py"
    )
    stopped_id = str(stopped[-1]["snapshot_id"])
    assert stopped[-1]["stage"] == "publish" and not (mr16_snapshots / stopped_id).exists()
    DatasetCommandExercise.run(["inspect", *mr16_common, "--snapshot-id", stopped_id], expected=4)
    DatasetCommandExercise.run(mr16_inspect)
    mr16_failures.append(
        ("Process death before rename", 75, "Snapshot unavailable; prior ID readable", "", "")
    )
    assert {p.name for p in mr16_snapshots.iterdir() if not p.name.startswith(".")} == {
        mr16_identity
    }
    recovered = DatasetCommandExercise.run(mr16_publish)
    assert recovered[-1]["snapshot_id"] == stopped_id
    DatasetCommandExercise.run(mr16_inspect)
    print("Retried publication and reopened prior snapshot:", stopped_id, mr16_identity)
finally:
    mr16_temporary.cleanup()
NotebookDisplay.show_table(
    ("Controlled failure", "Exit", "Outcome", "Invalid", "Gaps"), mr16_failures
)

Retried publication and reopened prior snapshot: 554722d42cd9f7536118c77e2973ff8457bb5da0d40d3288469c69f5075bc9fc 496c64f5376dc9b495cf7e8def9d1761a75c6cb7486a0b9578f9028a7a1349df


Controlled failure,Exit,Outcome,Invalid,Gaps
Wrong CSV schema,3,rejected,1,0
Wrong archive instrument,3,rejected,1,0
Wrong UTC day,3,rejected,1,0
Off-grid price units,3,rejected,1,0
Required coverage with a gap,3,rejected,0,1
Process death before rename,75,Snapshot unavailable; prior ID readable,,


**Interpretation and limits:** all invalid inputs fail with observed invalid counts;
the required coverage case is rejected with its gap count. None exposes an accepted
snapshot. Process death leaves hidden staging files but no partial selected ID; the
retry publishes a complete snapshot and the prior one remains readable. Persistence
or interruption errors handled by the command carry an explicit stage, candidate ID
when known, and recovery guidance. Abrupt process death cannot emit a terminal record.

The command accepts complete single-instrument/day acquisition manifests (several
may be listed); arbitrary overlaps or multi-instrument acquisition splitting are
unsupported. Both stores must remain available. Inspection revalidates all bytes
and admission, with bounded in-memory processing rather than a query API. The
[MR 16 record](../docs/roadmap/mr-16-implementation.md) separates installed synthetic
checks from the retained permitted sample and its unresolved historical evidence.
No M1 completion or P1 result is claimed.

**Next research question — MR 17:** which event revisions and eligibility assertions
were known at a chosen decision cutoff? Section 17 below now exercises that reader.


## 17. What was knowable at this decision cutoff?

**Question:** can a late publication, correction, duplicate capture or membership
change alter an earlier decision's inputs? We now use the installed `AsOfSelector`
for the retained synthetic trade histories. Prices, capture clocks, source positions
and correction evidence come from the earlier sections; they are engineering inputs.

The business window is half-open and uses the point's original `valid_from`
(capture for these Tardis points). Knowledge is a separate inclusive cutoff. A
cutoff projection must never replace the retained history used for earlier cutoffs.


In [27]:
# project library imports
from alpha_research.data.asof import AsOfSelector

mr17_start = datetime(2024, 6, 1, tzinfo=UTC)
mr17_binding = (
    captured_binding.instrument.source,
    captured_binding.manifest.job.stream,
    captured_binding.instrument.instrument_id,
)
mr17_report = mr14_policy.validate(
    captured_binding,
    tuple(captured_versions),
    eligibility=mr13_captured_history,
    corrections=(correction_evidence,),
)
mr17_selector = AsOfSelector((mr17_report.require_canonical(),), precedence=(mr17_binding,))
mr17_cutoffs = (
    mr17_start,
    mr17_start + timedelta(seconds=1),
    correction_evidence.ts_published,
    correction_evidence.ts_available,
    mr12_correction.ts_knowledge,
)
mr17_rows = []
for cutoff in mr17_cutoffs:
    selected = mr17_selector.select(
        (mr17_binding[2],), start=mr17_start, end=captured_binding.end, known_at=cutoff
    )
    mr17_rows.append(
        (
            cutoff,
            [str(row.occurrences[0].event.price) for row in selected],
            [row.ts_knowledge for row in selected],
        )
    )
assert [row[1] for row in mr17_rows] == [
    [],
    ["67540.01000000"],
    ["67540.01000000"],
    ["67541.01000000"],
    [],
]
mr17_earlier = mr17_selector.select(
    (mr17_binding[2],),
    start=mr17_start,
    end=captured_binding.end,
    known_at=mr17_start + timedelta(seconds=1),
)
assert mr17_earlier == (original_version,)
assert mr17_selector.history == (mr17_report.require_canonical(),)
NotebookDisplay.show_table(("Decision cutoff", "Selected prices", "Effective knowledge"), mr17_rows)
print("Retained original capture:", original_version.occurrences[0].event.provenance.ts_knowledge)
print(
    "Late publication / availability:",
    correction_evidence.ts_published,
    correction_evidence.ts_available,
)
print("Quality scope:", mr17_report.scope)

Decision cutoff,Selected prices,Effective knowledge
2024-06-01 00:00:00+00:00,[],[]
2024-06-01 00:00:01+00:00,['67540.01000000'],"[datetime.datetime(2024, 6, 1, 0, 0, 0, 4103, tzinfo=datetime.timezone.utc)]"
2024-06-01 00:00:09+00:00,['67540.01000000'],"[datetime.datetime(2024, 6, 1, 0, 0, 0, 4103, tzinfo=datetime.timezone.utc)]"
2024-06-01 00:00:10+00:00,['67541.01000000'],"[datetime.datetime(2024, 6, 1, 0, 0, 10, tzinfo=datetime.timezone.utc)]"
2024-06-01 12:00:00+00:00,[],[]


Retained original capture: 2024-06-01 00:00:00.004103+00:00
Late publication / availability: 2024-06-01 00:00:09+00:00 2024-06-01 00:00:10+00:00
Quality scope: exploratory_degraded


**Observed result:** publication alone does not admit the correction. The old price
survives until usable availability; the later membership correction withdraws this
business interval. Querying the earlier cutoff again still reconstructs the old price.

**Next check:** retain equal-capture source order and the correct subscription's
instrument-less disconnect. A query beginning after a known disconnect still needs
that boundary state; another instrument's disconnect must not affect its inputs.


In [28]:
mr17_payloads = []
mr17_precedence = []
mr17_orders = []
with TemporaryDirectory(prefix="nb17-controls-") as temporary:
    path = Path(temporary) / "frames.txt"
    for index, request in enumerate(SourceRequest.partition(captured_run)[:2]):
        symbol = request.instrument.symbol
        lines = [
            original_line.replace(b'"t":7', f'"t":{trade_id}'.encode())
            .replace(b"btcusdt", symbol.lower().encode())
            .replace(b"BTCUSDT", symbol.encode())
            for trade_id in (99, 1)
        ]
        lines.append(b"\n")
        path.write_bytes(b"".join(lines))
        origin = RawObject.from_file(
            path,
            request=request,
            retrieval_reference="synthetic:nb17:controls",
            ts_received=datetime(2026, 9, 17, tzinfo=UTC),
            ts_ingested=datetime(2026, 9, 17, tzinfo=UTC),
        )
        decoder = TardisDecoder(
            BinancePayloadDecoder(captured_run.instruments), request.instrument.source
        )
        items = []
        for position, raw in enumerate(lines):
            event = decoder.decode_line(
                raw.decode().rstrip("\n"),
                source_sequence=position,
                local_received=origin.ts_received,
                local_ingested=origin.ts_ingested,
                source_digest=origin.sha256,
                disconnect_capture_time=mr17_start + timedelta(seconds=1),
            )
            items.append(
                Occurrence(request=request, origin=origin, position=position, raw=raw, event=event)
            )
        report = mr14_policy.validate(request, items, eligibility=mr13_captured_history)
        mr17_payloads.append(report.require_canonical())
        mr17_precedence.append(
            (
                request.instrument.source,
                request.manifest.job.stream,
                request.instrument.instrument_id,
            )
        )
mr17_controls_selector = AsOfSelector(tuple(mr17_payloads), precedence=tuple(mr17_precedence))
mr17_controls = mr17_controls_selector.select(
    (mr17_binding[2],),
    start=mr17_start,
    end=captured_binding.end,
    known_at=mr17_start + timedelta(seconds=1),
)
assert [row.key[-1] for row in mr17_controls if isinstance(row, TradeRevision)] == [99, 1]
assert len(mr17_controls) == 3
mr17_boundary = mr17_controls_selector.select(
    (mr17_binding[2],),
    start=mr17_start + timedelta(seconds=2),
    end=captured_binding.end,
    known_at=mr17_start + timedelta(seconds=1),
)
assert len(mr17_boundary) == 1 and isinstance(mr17_boundary[0].event, Disconnect)
assert mr17_boundary[0].request.instrument.instrument_id == mr17_binding[2]
NotebookDisplay.show_table(
    ("Economic order", "Selected binding", "Source position", "Effective knowledge"),
    [
        (
            row.key[-1] if isinstance(row, TradeRevision) else "disconnect",
            row.occurrences[0].request.instrument.symbol
            if isinstance(row, TradeRevision)
            else row.request.instrument.symbol,
            row.occurrences[0].position if isinstance(row, TradeRevision) else row.position,
            row.ts_knowledge
            if isinstance(row, TradeRevision)
            else row.event.provenance.ts_knowledge,
        )
        for row in mr17_controls
    ],
)
print("Window begins after disconnect; retained boundary controls:", len(mr17_boundary))
print(
    "Unrelated ETH controls selected:",
    sum(
        isinstance(row, Occurrence) and row.request.instrument.symbol == "ETHUSDT"
        for row in mr17_controls
    ),
)

Economic order,Selected binding,Source position,Effective knowledge
99,BTCUSDT,0,2024-06-01 00:00:00.004103+00:00
1,BTCUSDT,1,2024-06-01 00:00:00.004103+00:00
disconnect,BTCUSDT,2,2024-06-01 00:00:01+00:00


Window begins after disconnect; retained boundary controls: 1
Unrelated ETH controls selected: 0


**Interpretation:** source positions preserve `[99, 1]` at equal capture times; no
trade-ID or content-hash sorting is used. The returned BTC control still has a null
instrument on the event, but its occurrence binds it to BTC. ETH's control is absent.
The selector retains controls; applying invalidation/cancellation belongs to later
observation and replay consumers.

**Snapshot check and required failures:** reopen an explicit published ID through
the same selector, inspect unknown archive availability, and reject malformed windows,
missing precedence, changed canonical claims and a mutable snapshot alias.


In [29]:
mr17_failures = []
with TemporaryDirectory(prefix="nb17-snapshot-") as temporary:
    local = Path(temporary)
    (local / "raw").mkdir()
    (local / "snapshots").mkdir()
    store = RawStore(local / "raw")
    source = local / "source.zip"
    source.write_bytes(mr14_raw_bytes)
    store.publish(source, descriptor=mr14_acquisition.completed[0], request=original_request)
    manifest, payloads = mr14_builds[0]
    snapshots = SnapshotStore(local / "snapshots", store)
    snapshot_id = snapshots.publish(manifest, payloads, max_bytes=262_144)
    archive_binding = (
        original_request.instrument.source,
        original_request.manifest.job.stream,
        original_request.instrument.instrument_id,
    )
    reader = AsOfSelector.read(
        local / "snapshots",
        store,
        identity=snapshot_id,
        max_bytes=262_144,
        precedence=(archive_binding,),
    )
    archive_selected = reader.select(
        (archive_binding[2],),
        start=mr17_start,
        end=original_request.manifest.job.end,
        known_at=mr17_start + timedelta(seconds=1),
    )
    assert archive_selected == () and b"original trade availability unknown" in reader.history[0]
    print("Verified snapshot:", snapshot_id)
    print(
        "Known archive events selected:",
        len(archive_selected),
        "— unknown capture/knowledge remains unknown; no event-time assumption.",
    )
    try:
        AsOfSelector.read(
            local / "snapshots",
            store,
            identity="latest",
            max_bytes=262_144,
            precedence=(archive_binding,),
        )
    except ValueError as error:
        mr17_failures.append(("Mutable snapshot ID", str(error)))
    else:
        raise AssertionError("mutable snapshot ID accepted")
for label, start, end, cutoff in (
    ("Empty business window", mr17_start, mr17_start, mr17_start),
    ("Naive cutoff", mr17_start, captured_binding.end, datetime(2024, 6, 1)),
):
    try:
        mr17_selector.select((mr17_binding[2],), start=start, end=end, known_at=cutoff)
    except (ValueError, TypeError) as error:
        mr17_failures.append((label, str(error)))
    else:
        raise AssertionError(f"invalid query accepted: {label}")
for label, payloads, precedence in (
    ("Missing binding precedence", mr17_selector.history, ()),
    (
        "Changed canonical quality",
        (mr17_selector.history[0].replace(b'"scope":"exploratory_degraded"', b'"scope":"valid"'),),
        (mr17_binding,),
    ),
):
    try:
        AsOfSelector(payloads, precedence=precedence)
    except ValueError as error:
        mr17_failures.append((label, str(error)))
    else:
        raise AssertionError(f"invalid input accepted: {label}")
assert len(mr17_failures) == 5
assert (
    mr17_selector.select(
        (mr17_binding[2],),
        start=mr17_start,
        end=captured_binding.end,
        known_at=mr17_start + timedelta(seconds=1),
    )
    == mr17_earlier
)
NotebookDisplay.show_table(("Required failure", "Observed rejection"), mr17_failures)

Verified snapshot: 496c64f5376dc9b495cf7e8def9d1761a75c6cb7486a0b9578f9028a7a1349df
Known archive events selected: 0 — unknown capture/knowledge remains unknown; no event-time assumption.


Required failure,Observed rejection
Mutable snapshot ID,snapshot requires an explicit lowercase SHA-256 identity
Empty business window,as-of selection requires a positive window and unique supported instruments
Naive cutoff,known_at must be a timezone-aware datetime
Missing binding precedence,precedence must name each retained input binding exactly once
Changed canonical quality,Invalid enum value 'valid' - at `$.scope`


**Limits:** this is a bounded in-memory reference selector over admitted individual
trades. Complete quality and eligibility histories remain in `selector.history`;
those retained inputs include future information and are not a cutoff projection.
An empty event result is not proof of valid coverage or an empty historical universe.
Unknown archive knowledge is never replaced by download time or an implicit
`event-time-assumption-v1`. Published input currently supports Binance archives;
the captured/control examples use synthetic admitted partitions, not published
Tardis or collector snapshots. The earlier OKX/Bitfinex exploration remains separate.

The [MR 17 record](../docs/roadmap/mr-17-implementation.md) records the installed
consumer, regressions and fresh-kernel evidence. These examples establish software
semantics, not empirical P1 membership, revision availability, fills or strategy results.

**Next research question — MR 19:** how can selected history be read in bounded
batches while retaining controls and earlier revisions? MR 18 is conditional on a
measured need after that interface exists. Neither package is implemented here.
